# Compare Step 2 Infill Models

This notebook compares the original paper-style `AudioMotionTransformer` mask infiller against our `AudioFIMCausalLM` on the same classic Step 2 windows:

- left token frame `t`
- predict middle token frames `t+1, t+2, t+3`
- right token frame `t+4`
- HuBERT continuous audio features sampled onto the motion-token frame timeline

Metrics included:

- token accuracy, exact frame accuracy, exact gap accuracy, per-quantizer accuracy
- decoded RVQVAE body-feature MAE/MSE for paired debugging
- paper-style full-sequence evaluator FID/diversity in the later section

The first half is for debugging. Use the full-sequence evaluator section for reportable FID/diversity.

In [ ]:
from pathlib import Path
import json
import math
import os
import sys
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

def find_project_root(start: Path = Path.cwd()) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "motion_generation").exists() and (path / "checkpoints").exists():
            return path
    raise RuntimeError("Could not find project root containing motion_generation/ and checkpoints/.")

PROJECT_ROOT = find_project_root()
MOTION_GENERATION_DIR = PROJECT_ROOT / "motion_generation"
EVALUATION_DIR = PROJECT_ROOT / "evaluation"
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(MOTION_GENERATION_DIR))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("project:", PROJECT_ROOT)
print("device:", DEVICE)

## Configuration

Keep `HISTORY_FRAMES_FOR_CAUSAL = 0` for the most apples-to-apples comparison, because the original mask transformer only sees left/right boundary frames. Raise it only if you intentionally want to test the causal model with extra history context.

In [ ]:
DATA_DIR = PROJECT_ROOT / "SuSuInterActs" / "SuSuInterActs"
MOTION_TOKEN_DIR = DATA_DIR / "motion_token_data"
EVAL_SPLIT_FILE = DATA_DIR / "split" / "val_file_list.txt"
MOTION2TEXT_JSON = DATA_DIR / "text_data" / "train.json"

MASK_CKPT = PROJECT_ROOT / "checkpoints" / "mask_transformer"
CAUSAL_CKPT = PROJECT_ROOT / "checkpoints" / "audio_fim_causal"

RVQVAE_CKPT = PROJECT_ROOT / "checkpoints" / "rvqvae" / "model" / "epoch_30.pth"
RVQVAE_MEAN_PATH = MOTION_GENERATION_DIR / "meta" / "mta_gen_demo" / "mean.npy"
RVQVAE_STD_PATH = MOTION_GENERATION_DIR / "meta" / "mta_gen_demo" / "std.npy"

# Source HuBERT feature fps and raw motion fps.
AUDIO_FPS = 50.0
RAW_MOTION_FPS = 20.0

# RVQVAE token frames are usually half the raw motion frame rate:
# raw 20 fps motion -> token 10 fps when unit_length=2.
MOTION_TOKEN_UNIT_LENGTH = 2.0
MOTION_TOKEN_FPS = RAW_MOTION_FPS / MOTION_TOKEN_UNIT_LENGTH

STEP = 4
HISTORY_FRAMES_FOR_CAUSAL = 0

# Increase these when the notebook flow is confirmed. Use None for full eval.
MAX_SEQUENCES = 128
MAX_WINDOWS = 512

# Old model masked-token generation. 1 is fastest; larger values fill masks in stages.
MASK_GENERATE_STEPS = 1

# Turn off if you only want token metrics or if RVQVAE files are unavailable.
DECODE_RVQVAE = True

# If True, clips invalid generated token IDs into 0..511 for RVQVAE decoding.
# Token metrics still count invalid IDs as wrong.
CLIP_INVALID_TOKENS_FOR_DECODE = True

AUDIO_FEAT_DIR = DATA_DIR / f"audio_features_hubert_layer9_fps{int(AUDIO_FPS)}"

for label, path in {
    "data": DATA_DIR,
    "motion tokens": MOTION_TOKEN_DIR,
    "audio features": AUDIO_FEAT_DIR,
    "eval split": EVAL_SPLIT_FILE,
    "mask checkpoint": MASK_CKPT,
    "causal checkpoint": CAUSAL_CKPT,
}.items():
    print(f"{label:18s}: {path} | exists={path.exists()}")

In [ ]:
import importlib

from safetensors import safe_open

from models.audio_motion_model import AudioMotionConfig, AudioMotionTransformer
from models.audio_fim_causal import AudioFIMCausalDataset, AudioFIMCausalLM
from scripts.train_audio_fim_causal import (
    discover_names,
    load_action_text_map,
    load_rvqvae_model_for_eval,
    load_sequences,
    motion_token_accuracy,
    read_split_file,
    decode_body_tokens_to_features,
)

def load_official_evaluator_helpers():
    """Load helpers from evaluation/evaluate_pred_motion_v2.py without
    leaving evaluation/ ahead of motion_generation/ on sys.path.
    """
    helper_names = [
        "calculate_alignment_single",
        "calculate_esd",
        "ChronTMR",
        "compute_128_sample_metrics",
        "compute_fid_diversity_metrics",
        "extract_audio_beats",
        "extract_motion_beats",
        "extract_motion_beats_for_esd",
        "PredMotionTextDataset",
    ]
    saved_path = list(sys.path)
    conflict_prefixes = ("models", "datasets", "evaluate_pred_motion_v2")
    saved_modules = {
        name: module
        for name, module in list(sys.modules.items())
        if name in conflict_prefixes or name.startswith("models.") or name.startswith("datasets.")
    }
    for name in list(saved_modules):
        sys.modules.pop(name, None)
    sys.path.insert(0, str(EVALUATION_DIR))
    try:
        module = importlib.import_module("evaluate_pred_motion_v2")
        return {name: getattr(module, name) for name in helper_names}
    finally:
        sys.path[:] = saved_path
        for name in list(sys.modules):
            if name in conflict_prefixes or name.startswith("models.") or name.startswith("datasets."):
                sys.modules.pop(name, None)
        sys.modules.update(saved_modules)

_eval_helpers = load_official_evaluator_helpers()
calculate_alignment_single = _eval_helpers["calculate_alignment_single"]
calculate_esd = _eval_helpers["calculate_esd"]
ChronTMR = _eval_helpers["ChronTMR"]
compute_128_sample_metrics = _eval_helpers["compute_128_sample_metrics"]
compute_fid_diversity_metrics = _eval_helpers["compute_fid_diversity_metrics"]
extract_audio_beats = _eval_helpers["extract_audio_beats"]
extract_motion_beats = _eval_helpers["extract_motion_beats"]
extract_motion_beats_for_esd = _eval_helpers["extract_motion_beats_for_esd"]
PredMotionTextDataset = _eval_helpers["PredMotionTextDataset"]

def load_mask_transformer_local(ckpt_path: Path, device: torch.device) -> AudioMotionTransformer:
    config = AudioMotionConfig.from_pretrained(str(ckpt_path))
    model = AudioMotionTransformer(config)
    safetensors_path = ckpt_path / "model.safetensors"
    bin_path = ckpt_path / "pytorch_model.bin"
    if safetensors_path.exists():
        state_dict = {}
        with safe_open(str(safetensors_path), framework="pt", device="cpu") as f:
            for key in f.keys():
                state_dict[key] = f.get_tensor(key)
    elif bin_path.exists():
        state_dict = torch.load(bin_path, map_location="cpu")
    else:
        raise FileNotFoundError(f"No model.safetensors or pytorch_model.bin in {ckpt_path}")
    model.load_state_dict(state_dict, strict=True)
    return model.to(device).eval()

def require_path(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")

## Load Eval Windows

The dataset object is reused for window construction. It samples HuBERT features at `token_frame_idx * AUDIO_FPS / MOTION_TOKEN_FPS`, so both models receive audio aligned to token frames.

In [ ]:
require_path(MOTION_TOKEN_DIR, "motion token dir")
require_path(AUDIO_FEAT_DIR, "audio feature dir")
require_path(EVAL_SPLIT_FILE, "eval split file")

action_text_map = load_action_text_map(str(MOTION2TEXT_JSON) if MOTION2TEXT_JSON.exists() else None)
split_names = read_split_file(str(EVAL_SPLIT_FILE))
eval_names = discover_names(MOTION_TOKEN_DIR, AUDIO_FEAT_DIR, split_names)

eval_sequences = load_sequences(
    eval_names,
    MOTION_TOKEN_DIR,
    AUDIO_FEAT_DIR,
    action_text_map,
    max_samples=MAX_SEQUENCES,
    audio_fps=AUDIO_FPS,
    motion_fps=RAW_MOTION_FPS,
    motion_token_fps=MOTION_TOKEN_FPS,
    motion_token_unit_length=MOTION_TOKEN_UNIT_LENGTH,
)

eval_dataset = AudioFIMCausalDataset(
    eval_sequences,
    step=STEP,
    audio_fps=AUDIO_FPS,
    motion_fps=MOTION_TOKEN_FPS,
    min_history_frames=HISTORY_FRAMES_FOR_CAUSAL,
    max_history_frames=HISTORY_FRAMES_FOR_CAUSAL,
    max_windows_per_sequence=None,
    seed=42,
)

num_windows = len(eval_dataset) if MAX_WINDOWS is None else min(int(MAX_WINDOWS), len(eval_dataset))
window_indices = np.linspace(0, len(eval_dataset) - 1, num_windows, dtype=np.int64).tolist()

print("sequences:", len(eval_sequences))
print("windows:", len(eval_dataset))
print("selected windows:", len(window_indices))
print("motion token fps:", MOTION_TOKEN_FPS)
print("audio fps:", AUDIO_FPS)

## Load Models

In [ ]:
require_path(MASK_CKPT, "mask transformer checkpoint")
require_path(CAUSAL_CKPT, "causal AudioFIM checkpoint")

mask_model = load_mask_transformer_local(MASK_CKPT, DEVICE)
causal_model = AudioFIMCausalLM.from_pretrained(str(CAUSAL_CKPT)).to(DEVICE).eval()

print("mask params:", sum(p.numel() for p in mask_model.parameters()))
print("causal params:", sum(p.numel() for p in causal_model.parameters()))

## Prediction Adapters

`mask_model` predicts all masked middle positions from `[left] [MASK x 12] [right]`.

`causal_model` sees its compact FIM prompt and autoregressively generates the middle tokens after the `[MIDDLE_MOTION]` marker.

In [ ]:
def tensor_to_numpy(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def audio5_from_example(example) -> np.ndarray:
    parts = [
        tensor_to_numpy(example.left_audio_feature),
        tensor_to_numpy(example.middle_audio_features),
        tensor_to_numpy(example.right_audio_feature),
    ]
    return np.concatenate([parts[0][None, :], parts[1], parts[2][None, :]], axis=0).astype(np.float32)

@torch.no_grad()
def predict_mask_middle(example) -> List[List[int]]:
    audio5 = audio5_from_example(example)
    ntpf = mask_model.config.num_tokens_per_frame
    codebook_size = mask_model.config.codebook_size
    offsets = [codebook_size * i for i in range(ntpf)]
    mask_token_id = mask_model.config.vocab_size - 1

    input_tokens = []
    for q in range(ntpf):
        input_tokens.append(int(example.left_anchor[q]) + offsets[q])
    input_tokens.extend([mask_token_id] * (3 * ntpf))
    for q in range(ntpf):
        input_tokens.append(int(example.right_anchor[q]) + offsets[q])

    input_ids = torch.tensor([input_tokens], dtype=torch.long, device=DEVICE)
    audio_feat = torch.tensor(audio5, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    output = mask_model.generate_sbs(input_ids, audio_feat, generate_steps=MASK_GENERATE_STEPS)
    output = output[0].detach().cpu().tolist()

    frames = []
    for frame_idx in range(1, 4):
        frame = []
        for q in range(ntpf):
            token_id = int(output[frame_idx * ntpf + q])
            frame.append(token_id - offsets[q])
        frames.append(frame)
    return frames

@torch.no_grad()
def predict_causal_middle(example) -> List[List[int]]:
    pred = causal_model.generate_infill(
        history_motion=example.history_motion,
        left_anchor=example.left_anchor,
        right_anchor=example.right_anchor,
        middle_audio_features=example.middle_audio_features,
        left_audio_feature=example.left_audio_feature,
        right_audio_feature=example.right_audio_feature,
        history_audio_features=example.history_audio_features,
        temperature=0.0,
    )
    return [list(map(int, frame)) for frame in pred]

def sanitize_tokens_for_decode(frames: Sequence[Sequence[int]], codebook_size: int = 512) -> List[List[int]]:
    arr = np.asarray(frames, dtype=np.int64)
    if CLIP_INVALID_TOKENS_FOR_DECODE:
        arr = np.clip(arr, 0, codebook_size - 1)
    return arr.tolist()

def invalid_token_fraction(frames: Sequence[Sequence[int]], codebook_size: int = 512) -> float:
    arr = np.asarray(frames, dtype=np.int64)
    return float(((arr < 0) | (arr >= codebook_size)).mean()) if arr.size else 0.0

## Metric Helpers

In [ ]:
def token_metrics(gt: Sequence[Sequence[int]], pred: Sequence[Sequence[int]]) -> Dict[str, float]:
    gt_arr = np.asarray(gt, dtype=np.int64)
    pred_arr = np.asarray(pred, dtype=np.int64)
    if gt_arr.shape != pred_arr.shape:
        return {"token_acc": np.nan, "exact_frame_acc": np.nan, "exact_gap_acc": 0.0}
    matches = gt_arr == pred_arr
    metrics = {
        "token_acc": float(matches.mean()),
        "exact_frame_acc": float(matches.all(axis=1).mean()),
        "exact_gap_acc": float(matches.all()),
    }
    for q in range(gt_arr.shape[1]):
        metrics[f"q{q + 1}_acc"] = float(matches[:, q].mean())
    return metrics

def decoded_feature_metrics(gt_feat: np.ndarray, pred_feat: np.ndarray) -> Dict[str, float]:
    diff = pred_feat.astype(np.float64) - gt_feat.astype(np.float64)
    vel_gt = np.diff(gt_feat, axis=0)
    vel_pred = np.diff(pred_feat, axis=0)
    vel_diff = vel_pred.astype(np.float64) - vel_gt.astype(np.float64)
    out = {
        "body_feat_mae": float(np.mean(np.abs(diff))),
        "body_feat_mse": float(np.mean(diff ** 2)),
        "body_feat_rmse": float(np.sqrt(np.mean(diff ** 2))),
    }
    if len(vel_diff) > 0:
        out["body_velocity_mse"] = float(np.mean(vel_diff ** 2))
    else:
        out["body_velocity_mse"] = np.nan
    return out

def sqrtm_psd(mat: torch.Tensor) -> torch.Tensor:
    mat = 0.5 * (mat + mat.T)
    vals, vecs = torch.linalg.eigh(mat)
    vals = torch.clamp(vals, min=0).sqrt()
    return (vecs * vals.unsqueeze(0)) @ vecs.T

def frechet_distance(real_features: np.ndarray, fake_features: np.ndarray, eps: float = 1e-5) -> float:
    real = torch.as_tensor(real_features, dtype=torch.float64)
    fake = torch.as_tensor(fake_features, dtype=torch.float64)
    if real.ndim != 2 or fake.ndim != 2:
        raise ValueError("features must be 2D: samples x dims")
    if real.shape[0] < 2 or fake.shape[0] < 2:
        return float("nan")
    mu1 = real.mean(dim=0)
    mu2 = fake.mean(dim=0)
    x1 = real - mu1
    x2 = fake - mu2
    cov1 = (x1.T @ x1) / max(1, real.shape[0] - 1)
    cov2 = (x2.T @ x2) / max(1, fake.shape[0] - 1)
    eye = torch.eye(cov1.shape[0], dtype=torch.float64)
    cov1 = cov1 + eps * eye
    cov2 = cov2 + eps * eye
    sqrt_cov1 = sqrtm_psd(cov1)
    covmean = sqrtm_psd(sqrt_cov1 @ cov2 @ sqrt_cov1)
    diff = mu1 - mu2
    fid = diff.dot(diff) + torch.trace(cov1 + cov2 - 2.0 * covmean)
    return float(torch.clamp(fid, min=0).item())

## Optional RVQVAE Decoder

This decodes RVQ token frames into normalized body feature vectors. The Frechet/FID-style metric below is computed in this decoded feature space.

In [ ]:
rvqvae = None
rvq_mean = None
rvq_std = None

if DECODE_RVQVAE:
    require_path(RVQVAE_CKPT, "RVQVAE checkpoint")
    require_path(RVQVAE_MEAN_PATH, "RVQVAE mean")
    require_path(RVQVAE_STD_PATH, "RVQVAE std")
    rvqvae, rvq_config = load_rvqvae_model_for_eval(RVQVAE_CKPT, DEVICE)
    rvq_mean = torch.tensor(np.load(RVQVAE_MEAN_PATH), dtype=torch.float32, device=DEVICE)
    rvq_std = torch.tensor(np.load(RVQVAE_STD_PATH), dtype=torch.float32, device=DEVICE)
    print("loaded RVQVAE:", RVQVAE_CKPT)
else:
    print("RVQVAE decoding disabled.")

@torch.no_grad()
def decode_body_features(frames: Sequence[Sequence[int]]) -> np.ndarray:
    if rvqvae is None:
        raise RuntimeError("RVQVAE decoding is disabled.")
    clean = sanitize_tokens_for_decode(frames)
    feat = decode_body_tokens_to_features(rvqvae, clean, rvq_mean, rvq_std, DEVICE)
    return feat.detach().cpu().numpy().astype(np.float32)

## Run Comparison

In [ ]:
rows = []

for idx in tqdm(window_indices, desc="compare windows"):
    example = eval_dataset[int(idx)]
    gt_middle = [list(map(int, frame)) for frame in example.middle_motion]
    predictions = {
        "mask": predict_mask_middle(example),
        "causal": predict_causal_middle(example),
    }

    decoded_gt_middle = None
    if DECODE_RVQVAE:
        decoded_gt_middle = decode_body_features(gt_middle)

    for model_name, pred_middle in predictions.items():
        row = {
            "model": model_name,
            "dataset_idx": int(idx),
            "name": example.name,
            "left_idx": int(example.left_idx),
            "right_idx": int(example.right_idx),
            "invalid_token_frac": invalid_token_fraction(pred_middle),
        }
        row.update(token_metrics(gt_middle, pred_middle))

        if DECODE_RVQVAE:
            pred_middle_clean = sanitize_tokens_for_decode(pred_middle)
            decoded_pred_middle = decode_body_features(pred_middle_clean)
            row.update(decoded_feature_metrics(decoded_gt_middle, decoded_pred_middle))

        rows.append(row)

results_df = pd.DataFrame(rows)
results_df.head()

## Aggregate Results

In [ ]:
mean_cols = [col for col in results_df.columns if col not in {"model", "name"}]
summary = results_df.groupby("model")[mean_cols].mean(numeric_only=True).T
display(summary)

## Inspect Best/Worst Windows

This is useful when the average metrics look okay but free-running examples still look wrong.

In [ ]:
sort_col = "token_acc"
for model_name in sorted(results_df["model"].unique()):
    print("\n", "=" * 80)
    print(model_name, "best")
    display(results_df[results_df.model == model_name].sort_values(sort_col, ascending=False).head(5))
    print(model_name, "worst")
    display(results_df[results_df.model == model_name].sort_values(sort_col, ascending=True).head(5))

## Save Metric Tables

## Paper-Style Full-Sequence Evaluator

The window-level FID above is only a lightweight diagnostic. This section builds full reconstructed clips and evaluates them with the repository's evaluator motion encoder from `checkpoints/eval_model/best_model.pt`.

Set `RUN_SEQUENCE_EXPORT = True` first to write full-sequence `.npy` files for GT, `codec_gt`, mask, and causal. Then set `RUN_EVALUATOR_FID = True` to compute evaluator-latent FID/diversity. The output format is compatible with `evaluation/datasets/pred_motion_dataset.py`.

`codec_gt` is decoded ground-truth RVQ tokens compared against the selected GT reference. When `OFFICIAL_GT_SOURCE = "raw_motion_data"`, `codec_gt` estimates the codec/tokenization/eval-prep floor before any Step 2 infill error is added.

In [ ]:
RUN_SEQUENCE_EXPORT = False
RUN_EVALUATOR_FID = False
RUN_BEAT_METRICS = False

# Use None for all eval sequences. Start with a small number while checking speed.
OFFICIAL_MAX_SEQUENCES = 128

# raw_motion_data is closer to final-motion evaluation; decoded_tokens is a cleaner codec-only baseline.
OFFICIAL_GT_SOURCE = "raw_motion_data"  # "raw_motion_data" or "decoded_tokens"

OFFICIAL_OUT_ROOT = MOTION_GENERATION_DIR / "notebooks" / "step2_official_eval_outputs"
OFFICIAL_GT_DIR = OFFICIAL_OUT_ROOT / "gt"
OFFICIAL_CODEC_GT_DIR = OFFICIAL_OUT_ROOT / "codec_gt"
OFFICIAL_MASK_DIR = OFFICIAL_OUT_ROOT / "mask"
OFFICIAL_CAUSAL_DIR = OFFICIAL_OUT_ROOT / "causal"

EVALUATOR_CKPT = PROJECT_ROOT / "checkpoints" / "eval_model" / "best_model.pt"
EVALUATOR_CFG_PATH = PROJECT_ROOT / "evaluation" / "config" / "train_bert_orig.yaml"
EVALUATOR_STATS_DIR = PROJECT_ROOT / "evaluation" / "stats" / "humanml3d" / "guoh3dfeats"
MOTION_DATA_DIR = DATA_DIR / "motion_data"
WAV_DIR = DATA_DIR / "wav_data"

print("output root:", OFFICIAL_OUT_ROOT)
print("evaluator ckpt:", EVALUATOR_CKPT, EVALUATOR_CKPT.exists())
print("evaluator cfg:", EVALUATOR_CFG_PATH, EVALUATOR_CFG_PATH.exists())
print("evaluator stats:", EVALUATOR_STATS_DIR, EVALUATOR_STATS_DIR.exists())

In [ ]:
def audio_feature_for_token_frame(item: Dict, token_frame_idx: int) -> torch.Tensor:
    audio = item["audio_features"]
    audio_fps = float(item.get("audio_fps", AUDIO_FPS))
    token_fps = float(item.get("motion_fps", MOTION_TOKEN_FPS))
    audio_idx = int(round(float(token_frame_idx) * audio_fps / token_fps))
    audio_idx = max(0, min(audio_idx, len(audio) - 1))
    return torch.tensor(audio[audio_idx], dtype=torch.float32)

def audio5_for_token_window(item: Dict, left_idx: int) -> np.ndarray:
    frames = [audio_feature_for_token_frame(item, left_idx + i).numpy() for i in range(STEP + 1)]
    return np.stack(frames, axis=0).astype(np.float32)

@torch.no_grad()
def predict_mask_window_from_item(item: Dict, left_idx: int) -> List[List[int]]:
    tokens = item["motion_tokens"]
    ntpf = mask_model.config.num_tokens_per_frame
    codebook_size = mask_model.config.codebook_size
    offsets = [codebook_size * i for i in range(ntpf)]
    mask_token_id = mask_model.config.vocab_size - 1
    right_idx = left_idx + STEP

    input_tokens = []
    for q in range(ntpf):
        input_tokens.append(int(tokens[left_idx][q]) + offsets[q])
    input_tokens.extend([mask_token_id] * ((STEP - 1) * ntpf))
    for q in range(ntpf):
        input_tokens.append(int(tokens[right_idx][q]) + offsets[q])

    input_ids = torch.tensor([input_tokens], dtype=torch.long, device=DEVICE)
    audio_feat = torch.tensor(audio5_for_token_window(item, left_idx), dtype=torch.float32, device=DEVICE).unsqueeze(0)
    output = mask_model.generate_sbs(input_ids, audio_feat, generate_steps=MASK_GENERATE_STEPS)[0].detach().cpu().tolist()

    pred = []
    for frame_idx in range(1, STEP):
        frame = []
        for q in range(ntpf):
            frame.append(int(output[frame_idx * ntpf + q]) - offsets[q])
        pred.append(frame)
    return pred

@torch.no_grad()
def predict_causal_window_from_item(item: Dict, left_idx: int) -> List[List[int]]:
    tokens = item["motion_tokens"]
    right_idx = left_idx + STEP
    history_start = max(0, left_idx - HISTORY_FRAMES_FOR_CAUSAL)
    history_indices = list(range(history_start, left_idx))
    middle_indices = list(range(left_idx + 1, right_idx))

    history_audio = None
    if history_indices:
        history_audio = torch.stack([audio_feature_for_token_frame(item, i) for i in history_indices], dim=0)

    return causal_model.generate_infill(
        history_motion=[list(tokens[i]) for i in history_indices],
        left_anchor=list(tokens[left_idx]),
        right_anchor=list(tokens[right_idx]),
        middle_audio_features=torch.stack([audio_feature_for_token_frame(item, i) for i in middle_indices], dim=0),
        left_audio_feature=audio_feature_for_token_frame(item, left_idx),
        right_audio_feature=audio_feature_for_token_frame(item, right_idx),
        history_audio_features=history_audio,
        temperature=0.0,
    )

def build_full_token_sequence(item: Dict, model_name: str) -> Tuple[List[List[int]], List[List[int]]]:
    tokens = item["motion_tokens"]
    left_indices = list(range(0, len(tokens) - STEP, STEP))
    if not left_indices:
        return [], []

    pred_full = []
    gt_full = []
    for window_idx, left_idx in enumerate(left_indices):
        right_idx = left_idx + STEP
        if model_name == "mask":
            middle = predict_mask_window_from_item(item, left_idx)
        elif model_name == "causal":
            middle = predict_causal_window_from_item(item, left_idx)
        else:
            raise ValueError(f"unknown model_name={model_name}")

        if window_idx == 0:
            pred_full.append(list(tokens[left_idx]))
            gt_full.append(list(tokens[left_idx]))
        pred_full.extend([list(frame) for frame in middle])
        pred_full.append(list(tokens[right_idx]))
        gt_full.extend([list(tokens[i]) for i in range(left_idx + 1, right_idx + 1)])

    return gt_full, pred_full

def decode_tokens_to_evaluator_body(tokens: Sequence[Sequence[int]]) -> np.ndarray:
    # RVQVAE decoder already restores token frames to source motion frames
    # (for this checkpoint: 10fps tokens -> 20fps body features).
    return decode_body_features(sanitize_tokens_for_decode(tokens))

def load_raw_gt_body(name: str, target_len: int) -> np.ndarray | None:
    path = MOTION_DATA_DIR / f"{name}.npy"
    if not path.exists():
        return None
    raw = np.load(path, allow_pickle=True)
    if isinstance(raw, np.ndarray) and raw.dtype == object:
        raw = raw.item()
    if not isinstance(raw, dict) or "body" not in raw:
        return None
    body = np.asarray(raw["body"], dtype=np.float32)
    return body[:target_len]

def save_evaluator_motion(path: Path, name: str, body: np.ndarray) -> None:
    body = np.asarray(body, dtype=np.float32)
    zeros_hand = np.zeros((len(body), 120), dtype=np.float32)
    payload = {"name": name, "body": body, "left": zeros_hand, "right": zeros_hand}
    np.save(path, payload)

def clean_eval_dir(path: Path, suffix: str) -> None:
    path.mkdir(parents=True, exist_ok=True)
    for old_file in path.glob(f"*_{suffix}.npy"):
        old_file.unlink()

In [ ]:
def export_full_sequence_eval_npys() -> pd.DataFrame:
    if not DECODE_RVQVAE:
        raise RuntimeError("Set DECODE_RVQVAE = True before exporting evaluator npys.")
    require_path(RVQVAE_CKPT, "RVQVAE checkpoint")

    selected_sequences = load_sequences(
        eval_names,
        MOTION_TOKEN_DIR,
        AUDIO_FEAT_DIR,
        action_text_map,
        max_samples=OFFICIAL_MAX_SEQUENCES,
        audio_fps=AUDIO_FPS,
        motion_fps=RAW_MOTION_FPS,
        motion_token_fps=MOTION_TOKEN_FPS,
        motion_token_unit_length=MOTION_TOKEN_UNIT_LENGTH,
    )

    clean_eval_dir(OFFICIAL_GT_DIR, "gt")
    clean_eval_dir(OFFICIAL_CODEC_GT_DIR, "pred")
    clean_eval_dir(OFFICIAL_MASK_DIR, "pred")
    clean_eval_dir(OFFICIAL_CAUSAL_DIR, "pred")

    rows = []
    for seq_idx, item in enumerate(tqdm(selected_sequences, desc="export full sequences")):
        gt_tokens, mask_tokens = build_full_token_sequence(item, "mask")
        _, causal_tokens = build_full_token_sequence(item, "causal")
        if not gt_tokens or not mask_tokens or not causal_tokens:
            continue

        mask_body = decode_tokens_to_evaluator_body(mask_tokens)
        causal_body = decode_tokens_to_evaluator_body(causal_tokens)
        decoded_gt_body = decode_tokens_to_evaluator_body(gt_tokens)
        decoded_frames = len(decoded_gt_body)
        target_len = min(len(mask_body), len(causal_body), decoded_frames)
        raw_gt = load_raw_gt_body(item["name"], target_len)
        raw_frames = 0 if raw_gt is None else len(raw_gt)

        if OFFICIAL_GT_SOURCE == "raw_motion_data":
            gt_body = raw_gt if raw_gt is not None and len(raw_gt) > 0 else decoded_gt_body
        elif OFFICIAL_GT_SOURCE == "decoded_tokens":
            gt_body = decoded_gt_body
        else:
            raise ValueError("OFFICIAL_GT_SOURCE must be raw_motion_data or decoded_tokens")

        target_len = min(len(gt_body), len(mask_body), len(causal_body))
        if target_len < 2:
            continue
        gt_body = gt_body[:target_len]
        codec_gt_body = decoded_gt_body[:target_len]
        mask_body = mask_body[:target_len]
        causal_body = causal_body[:target_len]

        file_stem = f"{seq_idx:06d}"
        save_evaluator_motion(OFFICIAL_GT_DIR / f"{file_stem}_gt.npy", item["name"], gt_body)
        save_evaluator_motion(OFFICIAL_CODEC_GT_DIR / f"{file_stem}_pred.npy", item["name"], codec_gt_body)
        save_evaluator_motion(OFFICIAL_MASK_DIR / f"{file_stem}_pred.npy", item["name"], mask_body)
        save_evaluator_motion(OFFICIAL_CAUSAL_DIR / f"{file_stem}_pred.npy", item["name"], causal_body)

        rows.append({
            "file_stem": file_stem,
            "name": item["name"],
            "frames_20fps": target_len,
            "token_frames": len(gt_tokens),
            "decoded_frames": decoded_frames,
            "raw_frames_before_trim": raw_frames,
            "gt_source": OFFICIAL_GT_SOURCE,
        })

    manifest = pd.DataFrame(rows)
    OFFICIAL_OUT_ROOT.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(OFFICIAL_OUT_ROOT / "manifest.csv", index=False)
    return manifest

if RUN_SEQUENCE_EXPORT:
    official_manifest = export_full_sequence_eval_npys()
    display(official_manifest.head())
    print("exported clips:", len(official_manifest))
else:
    print("Set RUN_SEQUENCE_EXPORT = True to export full-sequence evaluator npys.")

In [ ]:
from types import SimpleNamespace

def load_yaml_namespace(path: Path):
    import yaml
    with open(path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
    def convert(value):
        if isinstance(value, dict):
            return SimpleNamespace(**{k: convert(v) for k, v in value.items()})
        return value
    return convert(data)

def evaluator_length_to_mask(lengths: torch.Tensor, max_length: int, device: torch.device) -> torch.Tensor:
    lengths = lengths.to(device=device, dtype=torch.long)
    positions = torch.arange(max_length, device=device).unsqueeze(0)
    return positions < lengths.unsqueeze(1)

def load_evaluator_motion_encoder(device: torch.device):
    from evaluation.models.actor import ACTORStyleEncoder
    cfg = load_yaml_namespace(EVALUATOR_CFG_PATH)
    encoder = ACTORStyleEncoder(
        cfg.motion_encoder.nfeats,
        cfg.motion_encoder.vae,
        cfg.motion_encoder.latent_dim,
        cfg.motion_encoder.ff_size,
        cfg.motion_encoder.num_layers,
        cfg.motion_encoder.num_heads,
        cfg.motion_encoder.dropout,
        cfg.motion_encoder.activation,
    )
    ckpt = torch.load(EVALUATOR_CKPT, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        ckpt = ckpt["state_dict"]
    motion_state = {
        key.replace("motion_encoder.", "", 1): value
        for key, value in ckpt.items()
        if key.startswith("motion_encoder.")
    }
    missing, unexpected = encoder.load_state_dict(motion_state, strict=False)
    if missing or unexpected:
        print("motion encoder missing keys:", missing)
        print("motion encoder unexpected keys:", unexpected)
    encoder.to(device).eval()
    return cfg, encoder

def load_evaluator_body(path: Path) -> Tuple[str, np.ndarray]:
    raw = np.load(path, allow_pickle=True)
    if isinstance(raw, np.ndarray) and raw.shape == ():
        raw = raw.item()
    if isinstance(raw, dict):
        return raw.get("name", path.stem), np.asarray(raw["body"], dtype=np.float32)
    return path.stem, np.asarray(raw, dtype=np.float32)

def normalize_pad_body_for_evaluator(body: np.ndarray, mean: torch.Tensor, std: torch.Tensor, max_len: int) -> Tuple[torch.Tensor, int]:
    body = np.asarray(body, dtype=np.float32)
    length = min(len(body), max_len)
    body = body[:length]
    body_tensor = torch.tensor(body, dtype=torch.float32)
    body_dim = body_tensor.shape[1]
    body_tensor = (body_tensor - mean[:body_dim]) / (std[:body_dim] + 1e-12)
    if length < max_len:
        pad = torch.zeros((max_len - length, body_dim), dtype=torch.float32)
        body_tensor = torch.cat([body_tensor, pad], dim=0)
    return body_tensor, length

@torch.no_grad()
def encode_evaluator_latents(motion_dir: Path, suffix: str, cfg, encoder, device: torch.device, batch_size: int = 64) -> Tuple[List[str], np.ndarray]:
    files = sorted(motion_dir.glob(f"*_{suffix}.npy"))
    if not files:
        raise FileNotFoundError(f"No *_{suffix}.npy files found in {motion_dir}")
    mean = torch.load(EVALUATOR_STATS_DIR / "mean.pt", map_location="cpu").float()
    std = torch.load(EVALUATOR_STATS_DIR / "std.pt", map_location="cpu").float()
    max_len = int(cfg.dataset.max_motion_length)

    names = []
    latents = []
    batch_motions = []
    batch_lengths = []
    for path in tqdm(files, desc=f"encode {motion_dir.name}"):
        name, body = load_evaluator_body(path)
        motion, length = normalize_pad_body_for_evaluator(body, mean, std, max_len)
        names.append(name)
        batch_motions.append(motion)
        batch_lengths.append(length)
        if len(batch_motions) >= batch_size:
            x = torch.stack(batch_motions, dim=0).to(device)
            lengths = torch.tensor(batch_lengths, dtype=torch.long, device=device)
            mask = evaluator_length_to_mask(lengths, max_len, device)
            encoded = encoder({"x": x, "mask": mask})[:, 0]
            latents.append(encoded.detach().cpu())
            batch_motions.clear()
            batch_lengths.clear()
    if batch_motions:
        x = torch.stack(batch_motions, dim=0).to(device)
        lengths = torch.tensor(batch_lengths, dtype=torch.long, device=device)
        mask = evaluator_length_to_mask(lengths, max_len, device)
        encoded = encoder({"x": x, "mask": mask})[:, 0]
        latents.append(encoded.detach().cpu())
    return names, torch.cat(latents, dim=0).numpy()

# FID/Diversity below intentionally calls compute_fid_diversity_metrics()
# imported from evaluation/evaluate_pred_motion_v2.py.

In [ ]:
if RUN_EVALUATOR_FID:
    require_path(EVALUATOR_CKPT, "evaluator checkpoint")
    require_path(EVALUATOR_CFG_PATH, "evaluator config")
    cfg_eval, motion_encoder = load_evaluator_motion_encoder(DEVICE)
    _, gt_latents = encode_evaluator_latents(OFFICIAL_GT_DIR, "gt", cfg_eval, motion_encoder, DEVICE)
    evaluator_rows = []
    for model_name, motion_dir in {
        "codec_gt": OFFICIAL_CODEC_GT_DIR,
        "mask": OFFICIAL_MASK_DIR,
        "causal": OFFICIAL_CAUSAL_DIR,
    }.items():
        _, pred_latents = encode_evaluator_latents(motion_dir, "pred", cfg_eval, motion_encoder, DEVICE)
        metrics = compute_fid_diversity_metrics(gt_latents, pred_latents, diversity_times=300)
        metrics = {key: round(float(value), 6) for key, value in metrics.items()}
        metrics["model"] = model_name
        metrics["num_clips"] = int(len(pred_latents))
        evaluator_rows.append(metrics)
    evaluator_df = pd.DataFrame(evaluator_rows).set_index("model")
    display(evaluator_df)
else:
    print("Set RUN_EVALUATOR_FID = True after exporting full-sequence npys.")

## Official R@K Semantic Retrieval

This uses the ChronAccRet text+motion evaluator. `real_gt` points at `OFFICIAL_GT_DIR`, so set `OFFICIAL_GT_SOURCE = "raw_motion_data"` and rerun the export cell when you want true raw-GT retrieval.

In [ ]:
RUN_RETRIEVAL_RK = False

# ChronAccRet's TextEncoder hardcodes the author's bert-base-chinese path.
# Set this to a local bert-base-chinese folder if your environment is offline.
EVALUATOR_TEXT_MODEL_PATH = os.environ.get("EVALUATOR_TEXT_MODEL_PATH", "bert-base-chinese")
EVALUATOR_MOTION2TEXT_JSON = DATA_DIR / "text_data" / "motion2text.json"

def resolve_hf_path(path_like) -> str:
    path = Path(str(path_like))
    return str(path) if path.exists() else str(path_like)

def load_chrontmr_retrieval_model(device: torch.device):
    from transformers import AutoTokenizer
    import transformers

    cfg = load_yaml_namespace(EVALUATOR_CFG_PATH)
    text_model_name = resolve_hf_path(EVALUATOR_TEXT_MODEL_PATH)
    cfg.model.text_model_name = text_model_name

    original_from_pretrained = transformers.AutoModel.from_pretrained

    def redirected_from_pretrained(model_name, *args, **kwargs):
        name = str(model_name)
        if name == "bert-base-chinese" or "bert-base-chinese" in name or name.startswith("/data/home/jinch/"):
            return original_from_pretrained(text_model_name, *args, **kwargs)
        return original_from_pretrained(model_name, *args, **kwargs)

    transformers.AutoModel.from_pretrained = redirected_from_pretrained
    try:
        model = ChronTMR(cfg, vae=False)
    except Exception as exc:
        raise RuntimeError(
            "Could not load the ChronAccRet text model. Set EVALUATOR_TEXT_MODEL_PATH "
            "to a local bert-base-chinese checkpoint folder, or allow Hugging Face loading."
        ) from exc
    finally:
        transformers.AutoModel.from_pretrained = original_from_pretrained

    ckpt = torch.load(EVALUATOR_CKPT, map_location=device)
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        ckpt = ckpt["state_dict"]
    missing, unexpected = model.load_state_dict(ckpt, strict=False)
    if missing or unexpected:
        print("ChronTMR missing keys:", missing[:10], "..." if len(missing) > 10 else "")
        print("ChronTMR unexpected keys:", unexpected[:10], "..." if len(unexpected) > 10 else "")
    model.to(device).eval()

    try:
        tokenizer = AutoTokenizer.from_pretrained(text_model_name)
    except Exception as exc:
        raise RuntimeError(
            "Could not load the tokenizer. Set EVALUATOR_TEXT_MODEL_PATH "
            "to a local bert-base-chinese checkpoint folder."
        ) from exc
    return cfg, model, tokenizer

def make_retrieval_dataset(cfg, motion_dir: Path, suffix: str):
    return PredMotionTextDataset(
        cfg=cfg,
        motion_dir=str(motion_dir),
        motion2text_path=str(EVALUATOR_MOTION2TEXT_JSON),
        stats_dir=str(EVALUATOR_STATS_DIR),
        motion_type=suffix,
    )

if RUN_RETRIEVAL_RK:
    require_path(EVALUATOR_CKPT, "evaluator checkpoint")
    require_path(EVALUATOR_CFG_PATH, "evaluator config")
    require_path(EVALUATOR_MOTION2TEXT_JSON, "motion2text json")
    cfg_rk, chrontmr_model, chrontmr_tokenizer = load_chrontmr_retrieval_model(DEVICE)
    retrieval_rows = []
    for model_name, motion_dir, suffix in [
        ("real_gt", OFFICIAL_GT_DIR, "gt"),
        ("codec_gt", OFFICIAL_CODEC_GT_DIR, "pred"),
        ("mask", OFFICIAL_MASK_DIR, "pred"),
        ("causal", OFFICIAL_CAUSAL_DIR, "pred"),
    ]:
        dataset_rk = make_retrieval_dataset(cfg_rk, motion_dir, suffix)
        if len(dataset_rk) == 0:
            print(f"skip {model_name}: no valid samples")
            continue
        import random
        random.seed(int(cfg_rk.train.seed))
        np.random.seed(int(cfg_rk.train.seed))
        torch.manual_seed(int(cfg_rk.train.seed))
        metrics = compute_128_sample_metrics(cfg_rk, chrontmr_model, dataset_rk, DEVICE, chrontmr_tokenizer)
        row = {"model": model_name, "num_clips": int(len(dataset_rk))}
        for key in ["t2m/R01", "t2m/R02", "t2m/R03", "t2m/R05", "t2m/R10", "t2m/MedR"]:
            row[key] = metrics.get(key, np.nan)
        retrieval_rows.append(row)
    retrieval_df = pd.DataFrame(retrieval_rows).set_index("model")
    display(retrieval_df)
else:
    print("Set RUN_RETRIEVAL_RK = True after exporting full-sequence npys.")

In [ ]:
def wav_path_for_name(name: str) -> Path:
    return WAV_DIR / f"{name}.wav"

def compute_exported_beat_metrics(motion_dir: Path, suffix: str = "pred", fps: int = 20, tolerance: float = 0.1) -> Dict[str, float]:
    bas_distances, bhr_scores, esd_scores = [], [], []
    used = 0
    for path in tqdm(sorted(motion_dir.glob(f"*_{suffix}.npy")), desc=f"beat {motion_dir.name}"):
        name, body = load_evaluator_body(path)
        wav_path = wav_path_for_name(name)
        if not wav_path.exists():
            continue
        audio_beats = extract_audio_beats(str(wav_path))
        motion_beats = extract_motion_beats(body, fps=fps)
        bas, bhr = calculate_alignment_single(motion_beats, audio_beats, tolerance=tolerance)
        esd = calculate_esd(audio_beats, extract_motion_beats_for_esd(body, fps=fps))
        if bas is not None and not np.isnan(bas):
            bas_distances.append(bas)
        if bhr is not None:
            bhr_scores.append(bhr)
        esd_scores.append(esd)
        used += 1
    return {
        "BAS_distance_lower_better": float(np.mean(bas_distances)) if bas_distances else float("nan"),
        "BHR_higher_better": float(np.mean(bhr_scores)) if bhr_scores else float("nan"),
        "ESD_lower_better": float(np.mean(esd_scores)) if esd_scores else float("nan"),
        "num_evaluated": used,
    }

if RUN_BEAT_METRICS:
    beat_rows = []
    for model_name, motion_dir, suffix in [
        ("gt", OFFICIAL_GT_DIR, "gt"),
        ("codec_gt", OFFICIAL_CODEC_GT_DIR, "pred"),
        ("mask", OFFICIAL_MASK_DIR, "pred"),
        ("causal", OFFICIAL_CAUSAL_DIR, "pred"),
    ]:
        metrics = compute_exported_beat_metrics(motion_dir, suffix=suffix, fps=20, tolerance=0.1)
        metrics["model"] = model_name
        beat_rows.append(metrics)
    beat_df = pd.DataFrame(beat_rows).set_index("model")
    display(beat_df)
else:
    print("Set RUN_BEAT_METRICS = True after exporting full-sequence npys.")

In [ ]:
SAVE_RESULTS = False
OUT_DIR = MOTION_GENERATION_DIR / "notebooks" / "step2_metric_outputs"

if SAVE_RESULTS:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(OUT_DIR / "window_metrics.csv", index=False)
    summary.to_csv(OUT_DIR / "summary_metrics.csv")
    print("saved to", OUT_DIR)
else:
    print("Set SAVE_RESULTS = True to write CSV outputs.")

## Step 2 Investigation Harness

This refactor layer answers the current Step 2 questions without disturbing the older mask-vs-causal debug flow above.

It adds:

- `mask`: the original paper Step 2 checkpoint/path: `[left] [MASK x 3 frames] [right]`, with no previous-token context
- `causal_history_kN`: the current causal history path, exported once for quality with KV cache enabled by default
- explicit latency rows for `causal_history_kN_cache` and `causal_history_kN_nocache`, so the old 10x causal-vs-mask claim can be rechecked after KV caching
- optional `mask_history_kN`: an out-of-distribution inference probe that prepends previous `N` token frames to the original checkpoint. This is not the paper model and should not replace a trained masked+history baseline.
- first-window and full-clip latency accounting. `first_window` is the local Step 2 piece of TTFF; `full_clip` is closer to throughput/whole-clip cost.
- manual TTFF slots for HuBERT first chunk, LLM prefill, and LLM decode-to-first-keyframes measured on serving hardware
- seam metrics at sparse keyframe boundaries: acceleration and jerk discontinuity in decoded body-feature space
- export/eval hooks for FID/diversity, R@K, ESD/beat metrics, and seam metrics

Important reading rule: `step2_share_known_local` only divides Step 2 infill by `(Step 2 infill + RVQVAE decode)`. It does not answer whether Step 2 is on the end-to-end TTFF critical path until the HuBERT/LLM columns are filled.

Run the load/config/model/RVQVAE cells above first, then flip the `INV_RUN_*` flags below.


In [ ]:
# -----------------------------
# Investigation configuration
# -----------------------------
# Paper Step 2 mask baseline stays at 0 history. Set causal history to the
# k you used or want to test for the currently training causal checkpoint.
INV_HISTORY_FRAMES_FOR_CAUSAL = 8
INV_CAUSAL_HISTORY_MODEL = f"causal_history_k{INV_HISTORY_FRAMES_FOR_CAUSAL}"
INV_CAUSAL_HISTORY_CACHE_MODEL = f"{INV_CAUSAL_HISTORY_MODEL}_cache"
INV_CAUSAL_HISTORY_NOCACHE_MODEL = f"{INV_CAUSAL_HISTORY_MODEL}_nocache"

# Profile both cached and uncached causal generation before quoting the old
# causal-vs-mask latency gap. Quality export keeps only the cached causal row
# because cached/uncached should be identical at temperature=0.
INV_PROFILE_CAUSAL_NOCACHE = True
INV_RUN_CAUSAL_CACHE_SANITY = True

# Paper-faithful mask default is 0. Set INV_ENABLE_OOD_MASK_HISTORY_PROBE=True
# and choose k>0 only when intentionally probing the original checkpoint OOD.
# A real masked+history baseline should be trained/fine-tuned, not inferred OOD.
INV_HISTORY_FRAMES_FOR_MASK = 0
INV_ENABLE_OOD_MASK_HISTORY_PROBE = False
INV_MASK_HISTORY_MODEL = f"mask_history_k{INV_HISTORY_FRAMES_FOR_MASK}"

# Keep these false until you want the section to execute.
INV_RUN_WINDOW_COMPARISON = True
INV_RUN_SEQUENCE_EXPORT = True
INV_RUN_TTFF_PROFILE = True
INV_RUN_SEAM_METRICS = True
INV_RUN_EVALUATOR_FID = True
INV_RUN_RETRIEVAL_RK = True
INV_RUN_BEAT_METRICS = True

INV_MAX_WINDOWS = 512
INV_MAX_SEQUENCES = 128
INV_PROFILE_MAX_SEQUENCES = 16
INV_CACHE_SANITY_MAX_WINDOWS = 32

# Fill these from your serving stack. If they remain NaN, the TTFF table still
# reports local Step 2/decode costs, but TTFF totals and TTFF shares stay NaN.
INV_TTFF_MANUAL_MS = {
    "hubert_first_chunk_ms": np.nan,
    "llm_prefill_ms": np.nan,
    "llm_decode_to_first_keyframes_ms": np.nan,
}

INV_OUT_ROOT = MOTION_GENERATION_DIR / "notebooks" / "step2_investigation_outputs"
INV_GT_DIR = INV_OUT_ROOT / "gt"
INV_CODEC_GT_DIR = INV_OUT_ROOT / "codec_gt"
INV_MODEL_DIRS = {
    "mask": INV_OUT_ROOT / "mask",
    INV_CAUSAL_HISTORY_MODEL: INV_OUT_ROOT / INV_CAUSAL_HISTORY_MODEL,
}
if INV_ENABLE_OOD_MASK_HISTORY_PROBE and INV_HISTORY_FRAMES_FOR_MASK > 0:
    INV_MODEL_DIRS[INV_MASK_HISTORY_MODEL] = INV_OUT_ROOT / INV_MASK_HISTORY_MODEL

INV_PROFILE_MODEL_NAMES = ["mask", INV_CAUSAL_HISTORY_CACHE_MODEL]
if INV_PROFILE_CAUSAL_NOCACHE:
    INV_PROFILE_MODEL_NAMES.append(INV_CAUSAL_HISTORY_NOCACHE_MODEL)
if INV_MASK_HISTORY_MODEL in INV_MODEL_DIRS:
    INV_PROFILE_MODEL_NAMES.append(INV_MASK_HISTORY_MODEL)

INV_HISTORY_FRAMES_FOR_DATASET = max(
    INV_HISTORY_FRAMES_FOR_MASK if INV_ENABLE_OOD_MASK_HISTORY_PROBE else 0,
    INV_HISTORY_FRAMES_FOR_CAUSAL,
)

inv_eval_dataset = None
if "eval_sequences" in globals():
    inv_eval_dataset = AudioFIMCausalDataset(
        eval_sequences,
        step=STEP,
        audio_fps=AUDIO_FPS,
        motion_fps=MOTION_TOKEN_FPS,
        min_history_frames=INV_HISTORY_FRAMES_FOR_DATASET,
        max_history_frames=INV_HISTORY_FRAMES_FOR_DATASET,
        max_windows_per_sequence=None,
        seed=43,
    )

print("paper Step 2 baseline: mask (no previous-token context)")
print("causal history model for export:", INV_CAUSAL_HISTORY_MODEL, "(KV cache default)")
print("causal latency profile rows:", ", ".join(INV_PROFILE_MODEL_NAMES))
print("OOD mask history probe:", INV_MASK_HISTORY_MODEL if INV_MASK_HISTORY_MODEL in INV_MODEL_DIRS else "disabled")
print("investigation dataset history frames:", INV_HISTORY_FRAMES_FOR_DATASET)
print("investigation windows:", None if inv_eval_dataset is None else len(inv_eval_dataset))
print("output root:", INV_OUT_ROOT)


In [ ]:
# -----------------------------
# Optional OOD history probe for the original Step 2 checkpoint
# -----------------------------
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple


def inv_tensor_to_numpy(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def inv_motion_frame_to_global_ids(frame: Sequence[int], offsets: Sequence[int]) -> List[int]:
    return [int(frame[q]) + int(offsets[q]) for q in range(len(offsets))]


def inv_global_ids_to_motion_frame(output: Sequence[int], frame_idx: int, offsets: Sequence[int]) -> List[int]:
    ntpf = len(offsets)
    return [int(output[frame_idx * ntpf + q]) - int(offsets[q]) for q in range(ntpf)]


def inv_slice_example_history(example, history_frames: int) -> Tuple[List[List[int]], Optional[torch.Tensor]]:
    history_frames = max(0, int(history_frames))
    if history_frames == 0 or not getattr(example, "history_motion", None):
        return [], None
    history_motion = [list(map(int, frame)) for frame in example.history_motion[-history_frames:]]
    history_audio = example.history_audio_features
    if history_audio is not None:
        history_audio = history_audio[-len(history_motion):]
    return history_motion, history_audio


def inv_audio_feature_for_token_frame(item: Dict, token_frame_idx: int) -> torch.Tensor:
    audio = item["audio_features"]
    audio_fps = float(item.get("audio_fps", AUDIO_FPS))
    token_fps = float(item.get("motion_fps", MOTION_TOKEN_FPS))
    audio_idx = int(round(float(token_frame_idx) * audio_fps / token_fps))
    audio_idx = max(0, min(audio_idx, len(audio) - 1))
    return torch.tensor(audio[audio_idx], dtype=torch.float32)


def inv_audio_for_token_indices(item: Dict, token_indices: Sequence[int]) -> np.ndarray:
    return np.stack([inv_audio_feature_for_token_frame(item, i).numpy() for i in token_indices], axis=0).astype(np.float32)


@torch.no_grad()
def inv_predict_mask_middle_from_parts(
    *,
    history_motion: Sequence[Sequence[int]],
    left_anchor: Sequence[int],
    right_anchor: Sequence[int],
    middle_count: int,
    audio_context: np.ndarray,
) -> List[List[int]]:
    ntpf = mask_model.config.num_tokens_per_frame
    codebook_size = mask_model.config.codebook_size
    offsets = [codebook_size * i for i in range(ntpf)]
    mask_token_id = mask_model.config.vocab_size - 1

    input_tokens = []
    for frame in history_motion:
        input_tokens.extend(inv_motion_frame_to_global_ids(frame, offsets))
    input_tokens.extend(inv_motion_frame_to_global_ids(left_anchor, offsets))
    input_tokens.extend([mask_token_id] * (middle_count * ntpf))
    input_tokens.extend(inv_motion_frame_to_global_ids(right_anchor, offsets))

    input_ids = torch.tensor([input_tokens], dtype=torch.long, device=DEVICE)
    audio_feat = torch.tensor(audio_context, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    output = mask_model.generate_sbs(input_ids, audio_feat, generate_steps=MASK_GENERATE_STEPS)[0].detach().cpu().tolist()

    left_frame_pos = len(history_motion)
    return [
        inv_global_ids_to_motion_frame(output, left_frame_pos + frame_offset, offsets)
        for frame_offset in range(1, middle_count + 1)
    ]


@torch.no_grad()
def inv_predict_mask_middle_example(example, history_frames: int = 0) -> List[List[int]]:
    history_motion, history_audio = inv_slice_example_history(example, history_frames)
    audio_parts = []
    if history_audio is not None and history_motion:
        audio_parts.append(inv_tensor_to_numpy(history_audio))
    audio_parts.append(inv_tensor_to_numpy(example.left_audio_feature)[None, :])
    audio_parts.append(inv_tensor_to_numpy(example.middle_audio_features))
    audio_parts.append(inv_tensor_to_numpy(example.right_audio_feature)[None, :])
    audio_context = np.concatenate(audio_parts, axis=0).astype(np.float32)
    return inv_predict_mask_middle_from_parts(
        history_motion=history_motion,
        left_anchor=example.left_anchor,
        right_anchor=example.right_anchor,
        middle_count=len(example.middle_motion),
        audio_context=audio_context,
    )


@torch.no_grad()
def inv_predict_mask_window_from_item(item: Dict, left_idx: int, history_frames: int = 0) -> List[List[int]]:
    tokens = item["motion_tokens"]
    right_idx = left_idx + STEP
    history_start = max(0, left_idx - max(0, int(history_frames)))
    history_indices = list(range(history_start, left_idx))
    middle_indices = list(range(left_idx + 1, right_idx))
    context_indices = history_indices + [left_idx] + middle_indices + [right_idx]
    return inv_predict_mask_middle_from_parts(
        history_motion=[tokens[i] for i in history_indices],
        left_anchor=tokens[left_idx],
        right_anchor=tokens[right_idx],
        middle_count=len(middle_indices),
        audio_context=inv_audio_for_token_indices(item, context_indices),
    )


@torch.no_grad()
def inv_predict_causal_middle_example(
    example,
    history_frames: int = INV_HISTORY_FRAMES_FOR_CAUSAL,
    use_cache: bool = True,
) -> List[List[int]]:
    history_motion, history_audio = inv_slice_example_history(example, history_frames)
    pred = causal_model.generate_infill(
        history_motion=history_motion,
        left_anchor=example.left_anchor,
        right_anchor=example.right_anchor,
        middle_audio_features=example.middle_audio_features,
        left_audio_feature=example.left_audio_feature,
        right_audio_feature=example.right_audio_feature,
        history_audio_features=history_audio,
        temperature=0.0,
        use_cache=use_cache,
    )
    return [list(map(int, frame)) for frame in pred]


@torch.no_grad()
def inv_predict_causal_window_from_item(
    item: Dict,
    left_idx: int,
    history_frames: int = INV_HISTORY_FRAMES_FOR_CAUSAL,
    use_cache: bool = True,
) -> List[List[int]]:
    tokens = item["motion_tokens"]
    right_idx = left_idx + STEP
    history_start = max(0, left_idx - max(0, int(history_frames)))
    history_indices = list(range(history_start, left_idx))
    middle_indices = list(range(left_idx + 1, right_idx))
    history_audio = None
    if history_indices:
        history_audio = torch.stack([inv_audio_feature_for_token_frame(item, i) for i in history_indices], dim=0)
    return causal_model.generate_infill(
        history_motion=[list(tokens[i]) for i in history_indices],
        left_anchor=list(tokens[left_idx]),
        right_anchor=list(tokens[right_idx]),
        middle_audio_features=torch.stack([inv_audio_feature_for_token_frame(item, i) for i in middle_indices], dim=0),
        left_audio_feature=inv_audio_feature_for_token_frame(item, left_idx),
        right_audio_feature=inv_audio_feature_for_token_frame(item, right_idx),
        history_audio_features=history_audio,
        temperature=0.0,
        use_cache=use_cache,
    )


def inv_causal_model_names() -> set:
    return {
        INV_CAUSAL_HISTORY_MODEL,
        INV_CAUSAL_HISTORY_CACHE_MODEL,
        INV_CAUSAL_HISTORY_NOCACHE_MODEL,
    }


def inv_model_uses_cache(model_name: str) -> bool:
    if model_name == INV_CAUSAL_HISTORY_NOCACHE_MODEL:
        return False
    return True


def inv_predict_window_for_model(item: Dict, left_idx: int, model_name: str) -> List[List[int]]:
    if model_name == "mask":
        return inv_predict_mask_window_from_item(item, left_idx, history_frames=0)
    if model_name == INV_MASK_HISTORY_MODEL:
        return inv_predict_mask_window_from_item(
            item,
            left_idx,
            history_frames=INV_HISTORY_FRAMES_FOR_MASK,
        )
    if model_name in inv_causal_model_names():
        return inv_predict_causal_window_from_item(
            item,
            left_idx,
            history_frames=INV_HISTORY_FRAMES_FOR_CAUSAL,
            use_cache=inv_model_uses_cache(model_name),
        )
    raise ValueError(f"unknown model_name={model_name}")


def inv_build_full_token_sequence(item: Dict, model_name: str) -> Tuple[List[List[int]], List[List[int]]]:
    tokens = item["motion_tokens"]
    left_indices = list(range(0, len(tokens) - STEP, STEP))
    if not left_indices:
        return [], []

    pred_full, gt_full = [], []
    for window_idx, left_idx in enumerate(left_indices):
        right_idx = left_idx + STEP
        middle = inv_predict_window_for_model(item, left_idx, model_name)

        if window_idx == 0:
            pred_full.append(list(tokens[left_idx]))
            gt_full.append(list(tokens[left_idx]))
        pred_full.extend([list(frame) for frame in middle])
        pred_full.append(list(tokens[right_idx]))
        gt_full.extend([list(tokens[i]) for i in range(left_idx + 1, right_idx + 1)])
    return gt_full, pred_full


In [ ]:
# -----------------------------
# Window-level history baseline table
# -----------------------------
def inv_run_window_comparison(max_windows: int = INV_MAX_WINDOWS) -> pd.DataFrame:
    dataset = inv_eval_dataset if inv_eval_dataset is not None else eval_dataset
    if len(dataset) == 0:
        raise RuntimeError("Run the notebook's eval window loading cell first.")
    count = min(int(max_windows), len(dataset))
    selected = np.linspace(0, len(dataset) - 1, count, dtype=np.int64).tolist()
    rows = []
    for idx in tqdm(selected, desc="investigation windows"):
        example = dataset[int(idx)]
        gt_middle = [list(map(int, frame)) for frame in example.middle_motion]
        predictions = {
            "mask": inv_predict_mask_middle_example(example, history_frames=0),
            INV_CAUSAL_HISTORY_MODEL: inv_predict_causal_middle_example(
                example,
                history_frames=INV_HISTORY_FRAMES_FOR_CAUSAL,
            ),
        }
        if INV_MASK_HISTORY_MODEL in INV_MODEL_DIRS:
            predictions[INV_MASK_HISTORY_MODEL] = inv_predict_mask_middle_example(
                example,
                history_frames=INV_HISTORY_FRAMES_FOR_MASK,
            )
        for model_name, pred_middle in predictions.items():
            if not pred_middle:
                continue
            row = {
                "model": model_name,
                "dataset_idx": int(idx),
                "name": example.name,
                "left_idx": int(example.left_idx),
                "history_frames": (
                    INV_HISTORY_FRAMES_FOR_MASK if model_name == INV_MASK_HISTORY_MODEL
                    else INV_HISTORY_FRAMES_FOR_CAUSAL if model_name == INV_CAUSAL_HISTORY_MODEL
                    else 0
                ),
                "invalid_token_frac": invalid_token_fraction(pred_middle),
            }
            row.update(token_metrics(gt_middle, pred_middle))
            rows.append(row)
    return pd.DataFrame(rows)


if INV_RUN_WINDOW_COMPARISON:
    inv_window_df = inv_run_window_comparison()
    inv_window_summary = inv_window_df.groupby("model").mean(numeric_only=True).T
    display(inv_window_summary)
    display(inv_window_df.head())
else:
    print("Set INV_RUN_WINDOW_COMPARISON = True to compare paper mask vs optional OOD history probe windows.")


In [ ]:
# -----------------------------
# Sequence export, TTFF, seam/FID/R@K/ESD
# -----------------------------
import time
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple


def inv_clean_eval_dir(path: Path, suffix: str) -> None:
    path.mkdir(parents=True, exist_ok=True)
    for old_file in path.glob(f"*_{suffix}.npy"):
        old_file.unlink()


def inv_save_evaluator_motion(path: Path, name: str, body: np.ndarray) -> None:
    body = np.asarray(body, dtype=np.float32)
    zeros_hand = np.zeros((len(body), 120), dtype=np.float32)
    np.save(path, {"name": name, "body": body, "left": zeros_hand, "right": zeros_hand})


def inv_load_raw_gt_body(name: str, target_len: int) -> Optional[np.ndarray]:
    path = MOTION_DATA_DIR / f"{name}.npy"
    if not path.exists():
        return None
    raw = np.load(path, allow_pickle=True)
    if isinstance(raw, np.ndarray) and raw.dtype == object:
        raw = raw.item()
    if not isinstance(raw, dict) or "body" not in raw:
        return None
    return np.asarray(raw["body"], dtype=np.float32)[:target_len]


def inv_decode_tokens_to_body(tokens: Sequence[Sequence[int]]) -> np.ndarray:
    if "decode_tokens_to_evaluator_body" in globals():
        return decode_tokens_to_evaluator_body(tokens)
    return decode_body_features(sanitize_tokens_for_decode(tokens))


def inv_export_full_sequences(max_sequences: int = INV_MAX_SEQUENCES) -> pd.DataFrame:
    if not DECODE_RVQVAE:
        raise RuntimeError("Set DECODE_RVQVAE = True before exporting.")
    selected_sequences = load_sequences(
        eval_names,
        MOTION_TOKEN_DIR,
        AUDIO_FEAT_DIR,
        action_text_map,
        max_samples=max_sequences,
        audio_fps=AUDIO_FPS,
        motion_fps=RAW_MOTION_FPS,
        motion_token_fps=MOTION_TOKEN_FPS,
        motion_token_unit_length=MOTION_TOKEN_UNIT_LENGTH,
    )

    inv_clean_eval_dir(INV_GT_DIR, "gt")
    inv_clean_eval_dir(INV_CODEC_GT_DIR, "pred")
    for path in INV_MODEL_DIRS.values():
        inv_clean_eval_dir(path, "pred")

    rows = []
    for seq_idx, item in enumerate(tqdm(selected_sequences, desc="investigation export")):
        gt_tokens = None
        pred_tokens_by_model = {}
        for model_name in INV_MODEL_DIRS:
            cur_gt, pred = inv_build_full_token_sequence(item, model_name)
            if gt_tokens is None:
                gt_tokens = cur_gt
            pred_tokens_by_model[model_name] = pred
        if not gt_tokens or any(not pred for pred in pred_tokens_by_model.values()):
            continue

        decoded_gt = inv_decode_tokens_to_body(gt_tokens)
        pred_body_by_model = {name: inv_decode_tokens_to_body(tokens) for name, tokens in pred_tokens_by_model.items()}
        target_len = min([len(decoded_gt), *[len(body) for body in pred_body_by_model.values()]])
        raw_gt = inv_load_raw_gt_body(item["name"], target_len)
        if OFFICIAL_GT_SOURCE == "raw_motion_data":
            gt_body = raw_gt if raw_gt is not None and len(raw_gt) > 0 else decoded_gt
        else:
            gt_body = decoded_gt
        target_len = min([len(gt_body), len(decoded_gt), *[len(body) for body in pred_body_by_model.values()]])
        if target_len < 2:
            continue

        file_stem = f"{seq_idx:06d}"
        inv_save_evaluator_motion(INV_GT_DIR / f"{file_stem}_gt.npy", item["name"], gt_body[:target_len])
        inv_save_evaluator_motion(INV_CODEC_GT_DIR / f"{file_stem}_pred.npy", item["name"], decoded_gt[:target_len])
        for model_name, body in pred_body_by_model.items():
            inv_save_evaluator_motion(INV_MODEL_DIRS[model_name] / f"{file_stem}_pred.npy", item["name"], body[:target_len])
        rows.append({"file_stem": file_stem, "name": item["name"], "frames_20fps": target_len, "token_frames": len(gt_tokens)})

    manifest = pd.DataFrame(rows)
    INV_OUT_ROOT.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(INV_OUT_ROOT / "manifest.csv", index=False)
    return manifest


def inv_sync_device() -> None:
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def inv_elapsed_ms(fn: Callable[[], Any]) -> Tuple[Any, float]:
    inv_sync_device()
    start = time.perf_counter()
    value = fn()
    inv_sync_device()
    return value, (time.perf_counter() - start) * 1000.0


def inv_summarize_ms(values: Sequence[float]) -> Dict[str, float]:
    arr = np.asarray(values, dtype=np.float64)
    if arr.size == 0:
        return {"mean_ms": np.nan, "p50_ms": np.nan, "p95_ms": np.nan}
    return {"mean_ms": float(arr.mean()), "p50_ms": float(np.percentile(arr, 50)), "p95_ms": float(np.percentile(arr, 95))}


def inv_build_first_window_token_sequence(item: Dict, model_name: str) -> Tuple[List[List[int]], List[List[int]]]:
    tokens = item["motion_tokens"]
    if len(tokens) <= STEP:
        return [], []
    left_idx = 0
    right_idx = left_idx + STEP
    middle = inv_predict_window_for_model(item, left_idx, model_name)
    gt = [list(tokens[i]) for i in range(left_idx, right_idx + 1)]
    pred = [list(tokens[left_idx]), *[list(frame) for frame in middle], list(tokens[right_idx])]
    return gt, pred


def inv_profile_model_names() -> List[str]:
    names = []
    for model_name in INV_PROFILE_MODEL_NAMES:
        if model_name not in names:
            names.append(model_name)
    return names


def inv_profile_ttff(max_sequences: int = INV_PROFILE_MAX_SEQUENCES) -> Tuple[pd.DataFrame, pd.DataFrame]:
    selected = eval_sequences[: max(1, int(max_sequences))]
    profile_builders = {
        "first_window": inv_build_first_window_token_sequence,
        "full_clip": inv_build_full_token_sequence,
    }
    rows = []
    for item in tqdm(selected, desc="profile Step2 local"):
        for profile_scope, builder in profile_builders.items():
            for model_name in inv_profile_model_names():
                (gt_tokens, pred_tokens), infill_ms = inv_elapsed_ms(
                    lambda item=item, model_name=model_name, builder=builder: builder(item, model_name)
                )
                if not pred_tokens:
                    continue
                _, decode_ms = inv_elapsed_ms(lambda pred_tokens=pred_tokens: inv_decode_tokens_to_body(pred_tokens))
                rows.append({
                    "model": model_name,
                    "profile_scope": profile_scope,
                    "name": item["name"],
                    "token_frames": len(pred_tokens),
                    "infill_ms": infill_ms,
                    "rvqvae_decode_ms": decode_ms,
                })
    detail = pd.DataFrame(rows)
    summary_rows = []
    if detail.empty:
        return detail, pd.DataFrame()

    manual_values = list(INV_TTFF_MANUAL_MS.values())
    manual_complete = all(not np.isnan(v) for v in manual_values)
    for (model_name, profile_scope), group in detail.groupby(["model", "profile_scope"]):
        row = {"model": model_name, "profile_scope": profile_scope, "num_clips": int(len(group))}
        row.update({f"infill_{k}": v for k, v in inv_summarize_ms(group["infill_ms"]).items()})
        row.update({f"rvqvae_decode_{k}": v for k, v in inv_summarize_ms(group["rvqvae_decode_ms"]).items()})
        row.update(INV_TTFF_MANUAL_MS)
        local_total = row["infill_mean_ms"] + row["rvqvae_decode_mean_ms"]
        row["known_local_total_ms"] = local_total
        row["step2_share_known_local"] = row["infill_mean_ms"] / local_total if local_total > 0 else np.nan
        row["manual_ttff_complete"] = bool(manual_complete)
        if profile_scope == "first_window" and manual_complete:
            row["ttff_total_ms"] = local_total + sum(manual_values)
            row["step2_share_ttff"] = row["infill_mean_ms"] / row["ttff_total_ms"] if row["ttff_total_ms"] > 0 else np.nan
        else:
            row["ttff_total_ms"] = np.nan
            row["step2_share_ttff"] = np.nan
        summary_rows.append(row)
    summary = pd.DataFrame(summary_rows).set_index(["model", "profile_scope"]).sort_index()
    return detail, summary


def inv_run_causal_cache_sanity(max_windows: int = INV_CACHE_SANITY_MAX_WINDOWS) -> pd.DataFrame:
    dataset = inv_eval_dataset if inv_eval_dataset is not None else eval_dataset
    if len(dataset) == 0:
        raise RuntimeError("Run the notebook's eval window loading cell first.")
    count = min(int(max_windows), len(dataset))
    selected = np.linspace(0, len(dataset) - 1, count, dtype=np.int64).tolist()
    rows = []
    for idx in tqdm(selected, desc="causal cache sanity"):
        example = dataset[int(idx)]
        cached, cached_ms = inv_elapsed_ms(
            lambda example=example: inv_predict_causal_middle_example(
                example,
                history_frames=INV_HISTORY_FRAMES_FOR_CAUSAL,
                use_cache=True,
            )
        )
        uncached, uncached_ms = inv_elapsed_ms(
            lambda example=example: inv_predict_causal_middle_example(
                example,
                history_frames=INV_HISTORY_FRAMES_FOR_CAUSAL,
                use_cache=False,
            )
        )
        rows.append({
            "dataset_idx": int(idx),
            "name": example.name,
            "left_idx": int(example.left_idx),
            "same_tokens": cached == uncached,
            "cached_ms": cached_ms,
            "uncached_ms": uncached_ms,
            "speedup_uncached_over_cached": uncached_ms / cached_ms if cached_ms > 0 else np.nan,
            "cached_invalid_token_frac": invalid_token_fraction(cached),
            "uncached_invalid_token_frac": invalid_token_fraction(uncached),
        })
    return pd.DataFrame(rows)


def inv_seam_boundary_indices(num_frames: int) -> List[int]:
    stride = max(1, int(round(float(STEP) * float(MOTION_TOKEN_UNIT_LENGTH))))
    return list(range(stride, int(num_frames), stride))


def inv_seam_discontinuity_metrics(body: np.ndarray) -> Dict[str, float]:
    body = np.asarray(body, dtype=np.float64)
    acc, jerk = [], []
    for b in inv_seam_boundary_indices(len(body)):
        if 2 <= b and b + 2 < len(body):
            left = body[b] - 2.0 * body[b - 1] + body[b - 2]
            right = body[b + 2] - 2.0 * body[b + 1] + body[b]
            acc.append(float(np.linalg.norm(right - left)))
        if 3 <= b and b + 3 < len(body):
            left = body[b] - 3.0 * body[b - 1] + 3.0 * body[b - 2] - body[b - 3]
            right = body[b + 3] - 3.0 * body[b + 2] + 3.0 * body[b + 1] - body[b]
            jerk.append(float(np.linalg.norm(right - left)))
    return {
        "seam_accel_l2_mean": float(np.mean(acc)) if acc else np.nan,
        "seam_accel_l2_p95": float(np.percentile(acc, 95)) if acc else np.nan,
        "seam_jerk_l2_mean": float(np.mean(jerk)) if jerk else np.nan,
        "seam_jerk_l2_p95": float(np.percentile(jerk, 95)) if jerk else np.nan,
        "seam_count": int(max(len(acc), len(jerk))),
    }


def inv_compute_exported_seam_metrics(motion_dir: Path, suffix: str = "pred") -> Dict[str, float]:
    rows = []
    for path in sorted(motion_dir.glob(f"*_{suffix}.npy")):
        _, body = load_evaluator_body(path)
        rows.append(inv_seam_discontinuity_metrics(body))
    if not rows:
        return {"seam_accel_l2_mean": np.nan, "seam_accel_l2_p95": np.nan, "seam_jerk_l2_mean": np.nan, "seam_jerk_l2_p95": np.nan, "seam_count": 0, "num_evaluated": 0}
    df = pd.DataFrame(rows)
    return {
        "seam_accel_l2_mean": float(df["seam_accel_l2_mean"].mean()),
        "seam_accel_l2_p95": float(df["seam_accel_l2_p95"].mean()),
        "seam_jerk_l2_mean": float(df["seam_jerk_l2_mean"].mean()),
        "seam_jerk_l2_p95": float(df["seam_jerk_l2_p95"].mean()),
        "seam_count": int(df["seam_count"].sum()),
        "num_evaluated": int(len(df)),
    }


if INV_RUN_SEQUENCE_EXPORT:
    inv_manifest = inv_export_full_sequences()
    display(inv_manifest.head())
    print("exported clips:", len(inv_manifest))
else:
    print("Set INV_RUN_SEQUENCE_EXPORT = True to export mask/history/causal full-sequence npys.")

if INV_RUN_CAUSAL_CACHE_SANITY:
    inv_cache_sanity_df = inv_run_causal_cache_sanity()
    display(inv_cache_sanity_df.agg({
        "same_tokens": "mean",
        "cached_ms": "mean",
        "uncached_ms": "mean",
        "speedup_uncached_over_cached": "mean",
    }))
    display(inv_cache_sanity_df.head())
else:
    print("Set INV_RUN_CAUSAL_CACHE_SANITY = True to compare cached vs uncached causal outputs/timing.")

if INV_RUN_TTFF_PROFILE:
    inv_ttff_detail_df, inv_ttff_summary_df = inv_profile_ttff()
    display(inv_ttff_summary_df)
else:
    print("Set INV_RUN_TTFF_PROFILE = True to profile first-window TTFF-local and full-clip Step2/decode latency.")

if INV_RUN_SEAM_METRICS:
    seam_targets = [("gt", INV_GT_DIR, "gt"), ("codec_gt", INV_CODEC_GT_DIR, "pred")]
    seam_targets.extend((name, path, "pred") for name, path in INV_MODEL_DIRS.items())
    inv_seam_df = pd.DataFrame([{**inv_compute_exported_seam_metrics(path, suffix), "model": name} for name, path, suffix in seam_targets]).set_index("model")
    display(inv_seam_df)
else:
    print("Set INV_RUN_SEAM_METRICS = True after export to compute acceleration/jerk seam discontinuity.")


In [ ]:
# -----------------------------
# Optional official evaluator metrics for investigation outputs
# -----------------------------
if INV_RUN_EVALUATOR_FID:
    require_path(EVALUATOR_CKPT, "evaluator checkpoint")
    require_path(EVALUATOR_CFG_PATH, "evaluator config")
    cfg_eval, motion_encoder = load_evaluator_motion_encoder(DEVICE)
    _, gt_latents = encode_evaluator_latents(INV_GT_DIR, "gt", cfg_eval, motion_encoder, DEVICE)
    rows = []
    for model_name, motion_dir in {"codec_gt": INV_CODEC_GT_DIR, **INV_MODEL_DIRS}.items():
        _, pred_latents = encode_evaluator_latents(motion_dir, "pred", cfg_eval, motion_encoder, DEVICE)
        metrics = compute_fid_diversity_metrics(gt_latents, pred_latents, diversity_times=300)
        rows.append({**{k: round(float(v), 6) for k, v in metrics.items()}, "model": model_name, "num_clips": len(pred_latents)})
    inv_fid_df = pd.DataFrame(rows).set_index("model")
    display(inv_fid_df)
else:
    print("Set INV_RUN_EVALUATOR_FID = True after investigation export.")

if INV_RUN_RETRIEVAL_RK:
    cfg_rk, chrontmr_model, chrontmr_tokenizer = load_chrontmr_retrieval_model(DEVICE)
    targets = [("real_gt", INV_GT_DIR, "gt"), ("codec_gt", INV_CODEC_GT_DIR, "pred")]
    targets.extend((name, path, "pred") for name, path in INV_MODEL_DIRS.items())
    rows = []
    for model_name, motion_dir, suffix in targets:
        dataset_rk = make_retrieval_dataset(cfg_rk, motion_dir, suffix)
        if len(dataset_rk) == 0:
            print(f"skip {model_name}: no valid samples")
            continue
        import random
        random.seed(int(cfg_rk.train.seed))
        np.random.seed(int(cfg_rk.train.seed))
        torch.manual_seed(int(cfg_rk.train.seed))
        metrics = compute_128_sample_metrics(cfg_rk, chrontmr_model, dataset_rk, DEVICE, chrontmr_tokenizer)
        row = {"model": model_name, "num_clips": int(len(dataset_rk))}
        for key in ["t2m/R01", "t2m/R02", "t2m/R03", "t2m/R05", "t2m/R10", "t2m/MedR"]:
            row[key] = metrics.get(key, np.nan)
        rows.append(row)
    inv_rk_df = pd.DataFrame(rows).set_index("model")
    display(inv_rk_df)
else:
    print("Set INV_RUN_RETRIEVAL_RK = True after investigation export.")

if INV_RUN_BEAT_METRICS:
    targets = [("gt", INV_GT_DIR, "gt"), ("codec_gt", INV_CODEC_GT_DIR, "pred")]
    targets.extend((name, path, "pred") for name, path in INV_MODEL_DIRS.items())
    rows = []
    for model_name, motion_dir, suffix in targets:
        metrics = compute_exported_beat_metrics(motion_dir, suffix=suffix, fps=20, tolerance=0.1)
        rows.append({**metrics, "model": model_name})
    inv_beat_df = pd.DataFrame(rows).set_index("model")
    display(inv_beat_df)
else:
    print("Set INV_RUN_BEAT_METRICS = True after investigation export.")


## Multipart Codec + Multipart Step 2 Mask Transformer

This section adds the newly trained multipart stack to the comparison:

- `multipart_rvqvae` codec: four independent per-part RVQVAEs (upper / lower / feet / hands), each 512 codes x 4 quantizers, so every token frame is 16 tokens
- `mask_multipart`: the bidirectional Step 2 mask transformer trained on those 16-token frames with `scripts/train_audio_mask_multipart.py`

It is organized in two independent studies:

1. **Codec-only FID/FGD** (next two cells after setup): decodes ground-truth tokens from the old codec and the multipart codec over the same clip set, and measures reconstruction MAE/MSE plus evaluator-latent Frechet distance (FID/FGD) against the same GT. This isolates how much loss each codec itself introduces, before any infill error. Only needs the codec checkpoints, not the Step 2 model.
2. **Multipart Step 2 infill** (remaining cells): window metrics, full-sequence export, evaluator FID, R@K, beat, and seam metrics for `mask_multipart`, mirroring the older `mask` / `causal` flow.

How to read the numbers against the older `mask` / `causal` rows:

- Token metrics are only comparable within one codec (different codecs produce different GT tokens). Use them to judge the multipart infiller against its own token GT.
- Decoded body-feature MAE/MSE and the full-sequence evaluator metrics (FID, R@K, beat, seam) are computed in the same decoded body-feature space against the same raw GT motion, so those rows ARE directly comparable across codecs.
- `mp_codec_gt` is ground-truth multipart tokens decoded back to motion: the new codec's reconstruction floor, the analogue of the old `codec_gt` row.

Run the setup cells at the top first (project root, config, model/evaluator helper definitions, the old RVQVAE cell with `DECODE_RVQVAE = True`). These cells reuse `load_mask_transformer_local`, `decoded_feature_metrics`, the evaluator encoder, retrieval, and beat helpers defined above. Combined tables with the old-codec rows are shown automatically when the earlier sections were run in the same session.

In [ ]:
# -----------------------------
# Multipart configuration
# -----------------------------
# Set MP_STEP2_CKPT to the --output_dir you used with
# scripts/train_audio_mask_multipart.py. It must contain config.json +
# model.safetensors (+ multipart_token_layout.json written at save time).
# Only the Step 2 infill cells need it; the codec-only study does not.
MP_STEP2_CKPT = PROJECT_ROOT / "checkpoints" / "mask_multipart"

# Per-part codec checkpoints from checkpoints/multipart_rvqvae.
MP_RVQ_ROOT = PROJECT_ROOT / "checkpoints" / "multipart_rvqvae"
MP_PART_CKPTS = {
    part: MP_RVQ_ROOT / f"rvq_{part}_512x4_bs256_cosine" / "model" / "best.pth"
    for part in ("upper", "lower", "feet", "hands")
}

MP_MOTION_TOKEN_DIR = DATA_DIR / "motion_token_data_multipart_512x4"

# train_audio_mask_multipart.py defaults to 10 fps audio features. Set this to
# whatever you actually trained the multipart Step 2 model with (50.0 if you
# reused the fps50 HuBERT features the older models use).
MP_AUDIO_FPS = 10.0
MP_AUDIO_FEAT_DIR = DATA_DIR / (
    "audio_features_hubert_layer9_fps"
    + (str(int(MP_AUDIO_FPS)) if float(MP_AUDIO_FPS).is_integer() else str(MP_AUDIO_FPS).replace(".", "p"))
)

MP_MAX_SEQUENCES = MAX_SEQUENCES
MP_MAX_WINDOWS = MAX_WINDOWS
MP_EXPORT_MAX_SEQUENCES = 128

# Codec-only FID/FGD study (old codec vs multipart codec, no infill model).
MP_RUN_CODEC_COMPARISON = True
CODEC_MAX_SEQUENCES = 128
CODEC_OUT_ROOT = MOTION_GENERATION_DIR / "notebooks" / "step2_codec_eval_outputs"
CODEC_GT_DIR = CODEC_OUT_ROOT / "gt"
CODEC_OLD_DIR = CODEC_OUT_ROOT / "codec_old"
CODEC_MP_DIR = CODEC_OUT_ROOT / "codec_mp"

# Multipart Step 2 infill flow.
MP_RUN_WINDOW_COMPARISON = True
MP_RUN_SEQUENCE_EXPORT = True
MP_RUN_EVALUATOR_FID = True
MP_RUN_RETRIEVAL_RK = True
MP_RUN_BEAT_METRICS = True
MP_RUN_SEAM_METRICS = True

MP_MODEL_NAME = "mask_multipart"
MP_OUT_ROOT = MOTION_GENERATION_DIR / "notebooks" / "step2_multipart_eval_outputs"
MP_GT_DIR = MP_OUT_ROOT / "gt"
MP_CODEC_GT_DIR = MP_OUT_ROOT / "mp_codec_gt"
MP_PRED_DIR = MP_OUT_ROOT / MP_MODEL_NAME

for label, path in {
    "mp step2 checkpoint": MP_STEP2_CKPT,
    "mp motion tokens": MP_MOTION_TOKEN_DIR,
    "mp audio features": MP_AUDIO_FEAT_DIR,
    **{f"mp {part} codec": ckpt for part, ckpt in MP_PART_CKPTS.items()},
}.items():
    print(f"{label:22s}: {path} | exists={path.exists()}")

In [ ]:
# -----------------------------
# Load the multipart part codecs (independent of the Step 2 checkpoint)
# -----------------------------
from scripts.export_multipart_motion_tokens import load_part_codec
from scripts.train_audio_mask_multipart import (
    load_manifest as mp_load_manifest,
    load_token_json as mp_load_token_json,
)
from utils.multipart_motion import canonicalize_body_root, merge_parts_to_legacy_motion

for part, ckpt in MP_PART_CKPTS.items():
    require_path(ckpt, f"multipart {part} codec checkpoint")
require_path(MP_MOTION_TOKEN_DIR, "multipart motion token dir")

mp_codecs = {part: load_part_codec(ckpt, DEVICE) for part, ckpt in MP_PART_CKPTS.items()}
for part, loaded in mp_codecs.items():
    if loaded.part != part:
        raise ValueError(f"{part} checkpoint loaded part '{loaded.part}'")
if len({c.codebook_size for c in mp_codecs.values()}) != 1 or len({c.num_quantizers for c in mp_codecs.values()}) != 1:
    raise ValueError("All part codecs must share codebook_size and num_quantizers")
mp_codebook_size = int(next(iter(mp_codecs.values())).codebook_size)
mp_num_quantizers = int(next(iter(mp_codecs.values())).num_quantizers)

mp_manifest = mp_load_manifest(MP_MOTION_TOKEN_DIR)
mp_part_order = [str(p) for p in (mp_manifest or {}).get("part_order", list(mp_codecs))]
mp_ntpf = len(mp_part_order) * mp_num_quantizers

@torch.no_grad()
def mp_decode_tokens_to_body(frames: Sequence[Sequence[int]]) -> np.ndarray:
    """Decode (T, 16) multipart token frames to (T * unit_length, 153) body features."""
    arr = np.asarray(frames, dtype=np.int64)
    if CLIP_INVALID_TOKENS_FOR_DECODE:
        arr = np.clip(arr, 0, mp_codebook_size - 1)
    part_feats = {}
    for part_idx, part in enumerate(mp_part_order):
        cols = arr[:, part_idx * mp_num_quantizers : (part_idx + 1) * mp_num_quantizers]
        idx = torch.tensor(cols, dtype=torch.long, device=DEVICE).unsqueeze(0)
        loaded = mp_codecs[part]
        rec = loaded.model.decode({part: idx})[part]
        rec = loaded.normalizer.denormalize_tensor(part, rec)
        part_feats[part] = rec.squeeze(0).float().detach().cpu().numpy()
    return merge_parts_to_legacy_motion(part_feats)["body"].astype(np.float32)

print("mp part order:", mp_part_order, "| tokens/frame:", mp_ntpf, "| codebook:", mp_codebook_size)

### Codec-Only FID/FGD: How Much Does Each Codec Lose?

This decodes ground-truth tokens (no infill model in the loop) from both codecs over the **same clips** and compares them against the **same GT**, so the two rows are directly comparable:

- `codec_old`: GT tokens from `motion_token_data` decoded with `checkpoints/rvqvae` (the existing `codec_gt` floor)
- `codec_mp`: GT tokens from `motion_token_data_multipart_512x4` decoded with the four part codecs

Two views of codec loss:

- **Reconstruction table**: body-feature MAE/MSE/RMSE and velocity MSE per clip against the raw GT with the root channel canonicalized to frame deltas (the codec contract), then averaged. This is the direct "how much does the codec distort the motion" number.
- **Evaluator FID/FGD table**: evaluator-latent Frechet distance and diversity against raw GT clips, matching how the full-sequence section scores the infill models. GT files follow the existing convention (raw `motion_data` bodies as stored).

Clips are the eval sequences loaded at the top (`MAX_SEQUENCES`) that also have multipart tokens and raw motion, capped at `CODEC_MAX_SEQUENCES`. Requires the old RVQVAE cell to have run with `DECODE_RVQVAE = True`.

In [ ]:
if MP_RUN_CODEC_COMPARISON:
    if rvqvae is None:
        raise RuntimeError("Old codec not loaded. Set DECODE_RVQVAE = True and rerun the RVQVAE cell.")
    require_path(EVALUATOR_CKPT, "evaluator checkpoint")
    require_path(EVALUATOR_CFG_PATH, "evaluator config")

    clean_eval_dir(CODEC_GT_DIR, "gt")
    clean_eval_dir(CODEC_OLD_DIR, "pred")
    clean_eval_dir(CODEC_MP_DIR, "pred")

    codec_rows = []
    codec_exported = 0
    codec_skipped = {"no_mp_tokens": 0, "no_raw_gt": 0, "too_short": 0}
    for item in tqdm(eval_sequences, desc="codec recon export"):
        if CODEC_MAX_SEQUENCES is not None and codec_exported >= CODEC_MAX_SEQUENCES:
            break
        name = item["name"]
        mp_payload = mp_load_token_json(MP_MOTION_TOKEN_DIR / f"{name}.json")
        mp_tokens = (mp_payload or {}).get("tokens")
        if not mp_tokens:
            codec_skipped["no_mp_tokens"] += 1
            continue
        mp_tokens = [list(map(int, frame)) for frame in mp_tokens]

        old_body = decode_body_features(item["motion_tokens"])
        mp_body = mp_decode_tokens_to_body(mp_tokens)
        raw_gt = load_raw_gt_body(name, min(len(old_body), len(mp_body)))
        if raw_gt is None or len(raw_gt) == 0:
            codec_skipped["no_raw_gt"] += 1
            continue
        target_len = min(len(raw_gt), len(old_body), len(mp_body))
        if target_len < 2:
            codec_skipped["too_short"] += 1
            continue

        raw_gt = raw_gt[:target_len]
        old_body_clip = old_body[:target_len]
        mp_body_clip = mp_body[:target_len]
        # Codecs emit frame-delta root; canonicalize GT the same way so the
        # reconstruction error is not dominated by absolute-root clips.
        raw_gt_delta, root_schema, _ = canonicalize_body_root(raw_gt)

        for codec_name, body in (("codec_old", old_body_clip), ("codec_mp", mp_body_clip)):
            row = {
                "codec": codec_name,
                "name": name,
                "frames_20fps": int(target_len),
                "root_schema": root_schema,
            }
            row.update(decoded_feature_metrics(raw_gt_delta, body))
            codec_rows.append(row)

        stem = f"{codec_exported:06d}"
        save_evaluator_motion(CODEC_GT_DIR / f"{stem}_gt.npy", name, raw_gt)
        save_evaluator_motion(CODEC_OLD_DIR / f"{stem}_pred.npy", name, old_body_clip)
        save_evaluator_motion(CODEC_MP_DIR / f"{stem}_pred.npy", name, mp_body_clip)
        codec_exported += 1

    print("codec clips exported:", codec_exported, "| skipped:", codec_skipped)
    codec_recon_df = pd.DataFrame(codec_rows)
    codec_recon_summary = codec_recon_df.groupby("codec").mean(numeric_only=True)
    print("Codec reconstruction error vs delta-root raw GT (lower is better):")
    display(codec_recon_summary)

    if "motion_encoder" not in globals() or "cfg_eval" not in globals():
        cfg_eval, motion_encoder = load_evaluator_motion_encoder(DEVICE)
    _, codec_gt_latents = encode_evaluator_latents(CODEC_GT_DIR, "gt", cfg_eval, motion_encoder, DEVICE)
    rows = []
    for codec_name, motion_dir in (("codec_old", CODEC_OLD_DIR), ("codec_mp", CODEC_MP_DIR)):
        _, pred_latents = encode_evaluator_latents(motion_dir, "pred", cfg_eval, motion_encoder, DEVICE)
        metrics = compute_fid_diversity_metrics(codec_gt_latents, pred_latents, diversity_times=300)
        rows.append({
            **{key: round(float(value), 6) for key, value in metrics.items()},
            "codec": codec_name,
            "num_clips": int(len(pred_latents)),
        })
    codec_fid_df = pd.DataFrame(rows).set_index("codec")
    print("Codec-only evaluator FID/FGD vs the same raw GT clips:")
    display(codec_fid_df)
else:
    print("Set MP_RUN_CODEC_COMPARISON = True to compare the two codecs' reconstruction floors.")

In [ ]:
# -----------------------------
# Load the multipart Step 2 mask transformer + eval windows
# -----------------------------
from scripts.train_audio_mask_multipart import (
    AudioMotionMaskDataset as MPMaskDataset,
    discover_names as mp_discover_names,
    load_sequences as mp_load_sequences,
    read_split_file as mp_read_split_file,
)

require_path(MP_STEP2_CKPT, "multipart step2 checkpoint")
require_path(MP_AUDIO_FEAT_DIR, "multipart audio feature dir")

mp_mask_model = load_mask_transformer_local(MP_STEP2_CKPT, DEVICE)

mp_layout_path = MP_STEP2_CKPT / "multipart_token_layout.json"
if mp_layout_path.exists():
    with open(mp_layout_path, "r", encoding="utf-8") as f:
        mp_ckpt_layout = json.load(f)
    if [str(p) for p in mp_ckpt_layout.get("part_order", mp_part_order)] != mp_part_order:
        raise ValueError(
            f"checkpoint part_order {mp_ckpt_layout.get('part_order')} != token dir part_order {mp_part_order}"
        )
if int(mp_mask_model.config.num_tokens_per_frame) != mp_ntpf:
    raise ValueError(
        f"model tokens/frame {mp_mask_model.config.num_tokens_per_frame} != "
        f"{len(mp_part_order)} parts x {mp_num_quantizers} quantizers = {mp_ntpf}"
    )
if int(mp_mask_model.config.codebook_size) != mp_codebook_size:
    raise ValueError(
        f"model codebook_size {mp_mask_model.config.codebook_size} != codec codebook_size {mp_codebook_size}"
    )

mp_eval_names = mp_discover_names(
    MP_MOTION_TOKEN_DIR,
    MP_AUDIO_FEAT_DIR,
    mp_read_split_file(EVAL_SPLIT_FILE),
)
mp_eval_sequences, mp_load_stats = mp_load_sequences(
    mp_eval_names,
    MP_MOTION_TOKEN_DIR,
    MP_AUDIO_FEAT_DIR,
    codebook_size=mp_codebook_size,
    num_tokens_per_frame=mp_ntpf,
    audio_fps=MP_AUDIO_FPS,
    source_motion_fps_fallback=RAW_MOTION_FPS,
    motion_token_fps_override=None,
    motion_token_unit_length_override=None,
    max_sequences=MP_MAX_SEQUENCES,
)
mp_eval_dataset = MPMaskDataset(mp_eval_sequences, step=STEP, seed=42)

print("mp mask params:", sum(p.numel() for p in mp_mask_model.parameters()))
print("mp sequences:", len(mp_eval_sequences), "| load stats:", mp_load_stats)
print("mp windows:", len(mp_eval_dataset))

In [ ]:
@torch.no_grad()
def mp_predict_mask_middle(motion_tokens: Sequence[Sequence[int]], audio_features: torch.Tensor) -> List[List[int]]:
    """Predict the STEP-1 middle frames of one [left, mid..., right] window.

    motion_tokens: (STEP+1, mp_ntpf) window including both anchors.
    audio_features: (STEP+1, audio_feat_dim) sampled on the token timeline.
    """
    offsets = [mp_codebook_size * slot for slot in range(mp_ntpf)]
    mask_token_id = int(mp_mask_model.config.vocab_size) - 1

    input_tokens = [int(motion_tokens[0][q]) + offsets[q] for q in range(mp_ntpf)]
    input_tokens.extend([mask_token_id] * ((STEP - 1) * mp_ntpf))
    input_tokens.extend(int(motion_tokens[STEP][q]) + offsets[q] for q in range(mp_ntpf))

    input_ids = torch.tensor([input_tokens], dtype=torch.long, device=DEVICE)
    audio_feat = audio_features.to(device=DEVICE, dtype=torch.float32).unsqueeze(0)
    output = mp_mask_model.generate_sbs(input_ids, audio_feat, generate_steps=MASK_GENERATE_STEPS)
    output = output[0].detach().cpu().tolist()

    return [
        [int(output[frame_idx * mp_ntpf + q]) - offsets[q] for q in range(mp_ntpf)]
        for frame_idx in range(1, STEP)
    ]

def mp_token_metrics(gt: Sequence[Sequence[int]], pred: Sequence[Sequence[int]]) -> Dict[str, float]:
    gt_arr = np.asarray(gt, dtype=np.int64)
    pred_arr = np.asarray(pred, dtype=np.int64)
    if gt_arr.shape != pred_arr.shape:
        return {"token_acc": np.nan, "exact_frame_acc": np.nan, "exact_gap_acc": 0.0}
    matches = gt_arr == pred_arr
    metrics = {
        "token_acc": float(matches.mean()),
        "exact_frame_acc": float(matches.all(axis=1).mean()),
        "exact_gap_acc": float(matches.all()),
    }
    for part_idx, part in enumerate(mp_part_order):
        part_cols = matches[:, part_idx * mp_num_quantizers : (part_idx + 1) * mp_num_quantizers]
        metrics[f"{part}_acc"] = float(part_cols.mean())
    for q in range(mp_num_quantizers):
        metrics[f"q{q + 1}_acc"] = float(matches[:, q::mp_num_quantizers].mean())
    return metrics

In [ ]:
if MP_RUN_WINDOW_COMPARISON:
    mp_num_windows = (
        len(mp_eval_dataset)
        if MP_MAX_WINDOWS is None
        else min(int(MP_MAX_WINDOWS), len(mp_eval_dataset))
    )
    mp_window_indices = np.linspace(0, len(mp_eval_dataset) - 1, mp_num_windows, dtype=np.int64).tolist()

    mp_rows = []
    for idx in tqdm(mp_window_indices, desc="multipart windows"):
        example = mp_eval_dataset[int(idx)]
        gt_middle = [list(map(int, frame)) for frame in example.motion_tokens[1:STEP]]
        pred_middle = mp_predict_mask_middle(example.motion_tokens, example.audio_features)

        row = {
            "model": MP_MODEL_NAME,
            "dataset_idx": int(idx),
            "name": example.name,
            "left_idx": int(example.left_idx),
            "right_idx": int(example.right_idx),
            "invalid_token_frac": invalid_token_fraction(pred_middle, codebook_size=mp_codebook_size),
        }
        row.update(mp_token_metrics(gt_middle, pred_middle))
        decoded_gt_middle = mp_decode_tokens_to_body(gt_middle)
        decoded_pred_middle = mp_decode_tokens_to_body(pred_middle)
        row.update(decoded_feature_metrics(decoded_gt_middle, decoded_pred_middle))
        mp_rows.append(row)

    mp_window_df = pd.DataFrame(mp_rows)
    mp_window_summary = mp_window_df.groupby("model").mean(numeric_only=True).T
    display(mp_window_summary)

    # Cross-codec side-by-side. Token metrics are within-codec; the decoded
    # body_feat_* columns are the directly comparable ones.
    if "results_df" in globals():
        shared_cols = [
            col
            for col in [
                "token_acc",
                "exact_frame_acc",
                "exact_gap_acc",
                "invalid_token_frac",
                "body_feat_mae",
                "body_feat_mse",
                "body_feat_rmse",
                "body_velocity_mse",
            ]
            if col in results_df.columns and col in mp_window_df.columns
        ]
        mp_combined_window_summary = pd.concat(
            [
                results_df.groupby("model")[shared_cols].mean(),
                mp_window_df.groupby("model")[shared_cols].mean(),
            ]
        ).T
        print("Cross-codec window summary (body_feat_* rows are comparable; token rows are per-codec):")
        display(mp_combined_window_summary)
else:
    print("Set MP_RUN_WINDOW_COMPARISON = True to run multipart window metrics.")

In [ ]:
def mp_audio_feature_for_token_frame(item: Dict, token_frame_idx: int) -> torch.Tensor:
    audio = item["audio_features"]
    audio_fps = float(item.get("audio_fps", MP_AUDIO_FPS))
    token_fps = float(item.get("motion_token_fps", MOTION_TOKEN_FPS))
    audio_idx = int(round(float(token_frame_idx) * audio_fps / token_fps))
    audio_idx = max(0, min(audio_idx, len(audio) - 1))
    return torch.tensor(audio[audio_idx], dtype=torch.float32)

@torch.no_grad()
def mp_predict_window_from_item(item: Dict, left_idx: int) -> List[List[int]]:
    tokens = item["motion_tokens"]
    window = [list(tokens[left_idx + i]) for i in range(STEP + 1)]
    audio = torch.stack(
        [mp_audio_feature_for_token_frame(item, left_idx + i) for i in range(STEP + 1)],
        dim=0,
    )
    return mp_predict_mask_middle(window, audio)

def mp_build_full_token_sequence(item: Dict) -> Tuple[List[List[int]], List[List[int]]]:
    tokens = item["motion_tokens"]
    left_indices = list(range(0, len(tokens) - STEP, STEP))
    if not left_indices:
        return [], []

    pred_full, gt_full = [], []
    for window_idx, left_idx in enumerate(left_indices):
        right_idx = left_idx + STEP
        middle = mp_predict_window_from_item(item, left_idx)
        if window_idx == 0:
            pred_full.append(list(tokens[left_idx]))
            gt_full.append(list(tokens[left_idx]))
        pred_full.extend([list(frame) for frame in middle])
        pred_full.append(list(tokens[right_idx]))
        gt_full.extend([list(tokens[i]) for i in range(left_idx + 1, right_idx + 1)])
    return gt_full, pred_full

def mp_export_full_sequences(max_sequences: int = MP_EXPORT_MAX_SEQUENCES) -> pd.DataFrame:
    selected_sequences, _ = mp_load_sequences(
        mp_eval_names,
        MP_MOTION_TOKEN_DIR,
        MP_AUDIO_FEAT_DIR,
        codebook_size=mp_codebook_size,
        num_tokens_per_frame=mp_ntpf,
        audio_fps=MP_AUDIO_FPS,
        source_motion_fps_fallback=RAW_MOTION_FPS,
        motion_token_fps_override=None,
        motion_token_unit_length_override=None,
        max_sequences=max_sequences,
    )

    clean_eval_dir(MP_GT_DIR, "gt")
    clean_eval_dir(MP_CODEC_GT_DIR, "pred")
    clean_eval_dir(MP_PRED_DIR, "pred")

    rows = []
    for seq_idx, item in enumerate(tqdm(selected_sequences, desc="multipart export")):
        gt_tokens, pred_tokens = mp_build_full_token_sequence(item)
        if not gt_tokens or not pred_tokens:
            continue

        decoded_gt_body = mp_decode_tokens_to_body(gt_tokens)
        pred_body = mp_decode_tokens_to_body(pred_tokens)
        target_len = min(len(decoded_gt_body), len(pred_body))
        raw_gt = load_raw_gt_body(item["name"], target_len)
        if OFFICIAL_GT_SOURCE == "raw_motion_data":
            gt_body = raw_gt if raw_gt is not None and len(raw_gt) > 0 else decoded_gt_body
        elif OFFICIAL_GT_SOURCE == "decoded_tokens":
            gt_body = decoded_gt_body
        else:
            raise ValueError("OFFICIAL_GT_SOURCE must be raw_motion_data or decoded_tokens")

        target_len = min(len(gt_body), len(decoded_gt_body), len(pred_body))
        if target_len < 2:
            continue

        file_stem = f"{seq_idx:06d}"
        save_evaluator_motion(MP_GT_DIR / f"{file_stem}_gt.npy", item["name"], gt_body[:target_len])
        save_evaluator_motion(MP_CODEC_GT_DIR / f"{file_stem}_pred.npy", item["name"], decoded_gt_body[:target_len])
        save_evaluator_motion(MP_PRED_DIR / f"{file_stem}_pred.npy", item["name"], pred_body[:target_len])
        rows.append({
            "file_stem": file_stem,
            "name": item["name"],
            "frames_20fps": target_len,
            "token_frames": len(gt_tokens),
            "gt_source": OFFICIAL_GT_SOURCE,
        })

    manifest = pd.DataFrame(rows)
    MP_OUT_ROOT.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(MP_OUT_ROOT / "manifest.csv", index=False)
    return manifest

if MP_RUN_SEQUENCE_EXPORT:
    mp_manifest = mp_export_full_sequences()
    display(mp_manifest.head())
    print("exported multipart clips:", len(mp_manifest))
else:
    print("Set MP_RUN_SEQUENCE_EXPORT = True to export multipart full-sequence npys.")

In [ ]:
mp_metric_targets = [
    ("mp_codec_gt", MP_CODEC_GT_DIR, "pred"),
    (MP_MODEL_NAME, MP_PRED_DIR, "pred"),
]

if MP_RUN_EVALUATOR_FID:
    require_path(EVALUATOR_CKPT, "evaluator checkpoint")
    require_path(EVALUATOR_CFG_PATH, "evaluator config")
    if "motion_encoder" not in globals() or "cfg_eval" not in globals():
        cfg_eval, motion_encoder = load_evaluator_motion_encoder(DEVICE)
    _, mp_gt_latents = encode_evaluator_latents(MP_GT_DIR, "gt", cfg_eval, motion_encoder, DEVICE)
    rows = []
    for model_name, motion_dir, suffix in mp_metric_targets:
        _, pred_latents = encode_evaluator_latents(motion_dir, suffix, cfg_eval, motion_encoder, DEVICE)
        metrics = compute_fid_diversity_metrics(mp_gt_latents, pred_latents, diversity_times=300)
        rows.append({
            **{key: round(float(value), 6) for key, value in metrics.items()},
            "model": model_name,
            "num_clips": int(len(pred_latents)),
        })
    mp_fid_df = pd.DataFrame(rows).set_index("model")
    display(mp_fid_df)
    for previous_name in ("inv_fid_df", "evaluator_df"):
        if previous_name in globals():
            print(f"Combined evaluator FID/diversity table ({previous_name} + multipart):")
            display(pd.concat([globals()[previous_name], mp_fid_df]))
            break
else:
    print("Set MP_RUN_EVALUATOR_FID = True after the multipart export.")

if MP_RUN_RETRIEVAL_RK:
    require_path(EVALUATOR_MOTION2TEXT_JSON, "motion2text json")
    if "chrontmr_model" not in globals() or "cfg_rk" not in globals():
        cfg_rk, chrontmr_model, chrontmr_tokenizer = load_chrontmr_retrieval_model(DEVICE)
    rows = []
    for model_name, motion_dir, suffix in [("mp_real_gt", MP_GT_DIR, "gt"), *mp_metric_targets]:
        dataset_rk = make_retrieval_dataset(cfg_rk, motion_dir, suffix)
        if len(dataset_rk) == 0:
            print(f"skip {model_name}: no valid samples")
            continue
        import random
        random.seed(int(cfg_rk.train.seed))
        np.random.seed(int(cfg_rk.train.seed))
        torch.manual_seed(int(cfg_rk.train.seed))
        metrics = compute_128_sample_metrics(cfg_rk, chrontmr_model, dataset_rk, DEVICE, chrontmr_tokenizer)
        row = {"model": model_name, "num_clips": int(len(dataset_rk))}
        for key in ["t2m/R01", "t2m/R02", "t2m/R03", "t2m/R05", "t2m/R10", "t2m/MedR"]:
            row[key] = metrics.get(key, np.nan)
        rows.append(row)
    mp_rk_df = pd.DataFrame(rows).set_index("model")
    display(mp_rk_df)
    if "inv_rk_df" in globals():
        print("Combined R@K table (investigation + multipart):")
        display(pd.concat([inv_rk_df, mp_rk_df]))
else:
    print("Set MP_RUN_RETRIEVAL_RK = True after the multipart export.")

In [ ]:
if MP_RUN_BEAT_METRICS:
    rows = []
    for model_name, motion_dir, suffix in [("mp_gt", MP_GT_DIR, "gt"), *mp_metric_targets]:
        metrics = compute_exported_beat_metrics(motion_dir, suffix=suffix, fps=20, tolerance=0.1)
        rows.append({**metrics, "model": model_name})
    mp_beat_df = pd.DataFrame(rows).set_index("model")
    display(mp_beat_df)
    if "inv_beat_df" in globals():
        print("Combined beat table (investigation + multipart):")
        display(pd.concat([inv_beat_df, mp_beat_df]))
else:
    print("Set MP_RUN_BEAT_METRICS = True after the multipart export.")

if MP_RUN_SEAM_METRICS:
    if "inv_compute_exported_seam_metrics" not in globals():
        print("Run the investigation harness cells above first to define the seam metric helpers.")
    else:
        seam_targets = [("mp_gt", MP_GT_DIR, "gt"), *mp_metric_targets]
        mp_seam_df = pd.DataFrame(
            [
                {**inv_compute_exported_seam_metrics(path, suffix), "model": name}
                for name, path, suffix in seam_targets
            ]
        ).set_index("model")
        display(mp_seam_df)
        if "inv_seam_df" in globals():
            print("Combined seam table (investigation + multipart):")
            display(pd.concat([inv_seam_df, mp_seam_df]))
else:
    print("Set MP_RUN_SEAM_METRICS = True after the multipart export.")

MP_SAVE_RESULTS = False
if MP_SAVE_RESULTS:
    MP_OUT_ROOT.mkdir(parents=True, exist_ok=True)
    for frame_name in (
        "codec_recon_df",
        "codec_recon_summary",
        "codec_fid_df",
        "mp_window_df",
        "mp_window_summary",
        "mp_combined_window_summary",
        "mp_fid_df",
        "mp_rk_df",
        "mp_beat_df",
        "mp_seam_df",
    ):
        if frame_name in globals():
            globals()[frame_name].to_csv(MP_OUT_ROOT / f"{frame_name}.csv")
            print("saved", MP_OUT_ROOT / f"{frame_name}.csv")
else:
    print("Set MP_SAVE_RESULTS = True to write the multipart metric CSVs.")